# 第2回・発展編：用語集を学習する方法と、検索する方法を比べる

講座の用語集 58 語から作った Q&A 116 件で LoRA 学習します。回答は普通の文章です。
**学習前・学習後とも、回答時のモデル入力は質問だけです。用語集・正解・採点用語句は渡しません。**

比較するのは、①元のモデル、②学習後のモデル、③用語集の文字列検索です。
検索版はモデルを使わず、`app/tools.py` の検索結果の先頭の定義をそのまま返します。

**使い方**: Colab の T4 カーネルを選び、カーネルを再起動してセクション 0〜7 を上から実行してください。
入力ファイルはこのノートブックに同梱済みです。VS Code の Colab 接続でも個別アップロードは不要です。
元ファイルは `lectures/02/advanced/glossary_sft.jsonl`、`data/glossary.json`、
`lectures/02/advanced/evaluation.json` です。同梱時点の内容と SHA-256 を結果に残します。

学習はまず **5 エポック**にしています。116 件あるため、図書館版の 20 エポックをそのまま使わず、
短い実験から始めるための設定です。性能や学習時間は実測してください。結果の保存先は `/content/glossary_lora_lab` です。

評価は 15 問です。「学習と同じ質問」6 問は記憶の確認、「言い換え」6 問は別の聞き方への対応、
「用語集にない語」3 問は既存の知識や断定の変化を観察します。未掲載でも正しく説明できれば正解です。

## 0. GPU とライブラリ

最初のセルで T4 が表示されることを確認します。ライブラリは元の演習と同じ版です。
インストール中の `cudf` / `pyarrow` の依存警告については元の演習と同じ扱いです。

In [34]:
import subprocess
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                         capture_output=True, text=True, check=True).stdout)
except (FileNotFoundError, subprocess.CalledProcessError):
    raise RuntimeError("Colab の T4 カーネルを選び、カーネルを再起動してください。")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [35]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## 1. 設定

5 エポック・量子化なしの LoRA です。回答は自然文なので、生成上限は 256 トークンにします。
学習前後の生成条件は同じです。設定を変えて再学習するときはカーネルを再起動してください。

In [36]:
from pathlib import Path
import hashlib, importlib.util, json, math, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 5,
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,
    "gen": {"do_sample": False},
    "work_dir": "/content/glossary_lora_lab",
    "drive_dir": "/content/drive/MyDrive/aie2026/glossary_lora_lab",
}
WORK = Path(CONFIG["work_dir"])
WORK.mkdir(parents=True, exist_ok=True)
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 5,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 256,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/glossary_lora_lab",
  "drive_dir": "/content/drive/MyDrive/aie2026/glossary_lora_lab"
}


## 2. 学習データ・評価問題・検索の準備

次の大きなセルは、元のファイルを Colab に持ち込むための同梱データです。
通常は編集せず実行してください。`inputs/` に展開し、ファイルのハッシュを確認します。
`glossary_sft.jsonl` は `scripts/make_sft_dataset.py` の出力そのものです。

学習時には質問と定義文の対を使います。評価問題やその正解は学習に追加しません。
ただし「学習と同じ質問」の 6 問は、意図的に学習問題を再利用しています。
言い換えと未掲載語の質問文は、116 件の学習問題と重複しないことを確認します。
未掲載語の参照回答の出典は `evaluation.json` に記録しています。

In [37]:
# 元ファイルの内容を文字列として同梱。ここではモデルに入力しません。
BUNDLED_FILES = json.loads(r'''{
  "glossary.json": "{\n  \"version\": \"2026-draft-1\",\n  \"note\": \"AIE2026 第1〜3回・第5回の用語集（たたき台）。講義スライド確定後に内容を差し替える。\",\n  \"terms\": [\n    {\n      \"id\": \"ai-engineering\",\n      \"term\": \"AIエンジニアリング\",\n      \"aliases\": [\"AI Engineering\"],\n      \"lecture\": 1,\n      \"category\": \"全体像\",\n      \"definition\": \"学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。\",\n      \"related\": [\"llm\", \"evaluation-loop\"]\n    },\n    {\n      \"id\": \"evaluation-loop\",\n      \"term\": \"評価と改善のサイクル\",\n      \"aliases\": [\"改善サイクル\"],\n      \"lecture\": 1,\n      \"category\": \"全体像\",\n      \"definition\": \"作る→測る→直すを繰り返す開発の進め方。AIアプリは出力が確率的に揺れるため、感覚ではなく評価データや指標で良し悪しを判断し、その結果をもとにプロンプト・データ・モデル・ツールを改善する。本講座では学習支援エージェントを題材に、回をまたいでこのサイクルを回す。\",\n      \"related\": [\"ai-engineering\", \"hallucination\"]\n    },\n    {\n      \"id\": \"llm\",\n      \"term\": \"LLM（大規模言語モデル）\",\n      \"aliases\": [\"大規模言語モデル\", \"Large Language Model\"],\n      \"lecture\": 1,\n      \"category\": \"LLM基礎\",\n      \"definition\": \"大量のテキストで学習し、入力に続くトークンを予測することで文章を生成するモデル。質問応答、要約、コード生成など多くのタスクを、指示（プロンプト）を変えるだけでこなせる。\",\n      \"related\": [\"token\", \"prompt\", \"pretraining\"]\n    },\n    {\n      \"id\": \"token\",\n      \"term\": \"トークン\",\n      \"aliases\": [\"token\"],\n      \"lecture\": 1,\n      \"category\": \"LLM基礎\",\n      \"definition\": \"LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。\",\n      \"related\": [\"context-window\", \"rate-limit\"]\n    },\n    {\n      \"id\": \"context-window\",\n      \"term\": \"コンテキストウィンドウ\",\n      \"aliases\": [\"コンテキスト長\", \"context window\"],\n      \"lecture\": 1,\n      \"category\": \"LLM基礎\",\n      \"definition\": \"LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。\",\n      \"related\": [\"token\", \"agent-loop\"]\n    },\n    {\n      \"id\": \"prompt\",\n      \"term\": \"プロンプト\",\n      \"aliases\": [\"prompt\"],\n      \"lecture\": 1,\n      \"category\": \"LLM基礎\",\n      \"definition\": \"LLMに渡す入力文。質問だけでなく、役割、制約、出力形式、例などを含めることで出力を方向づけられる。\",\n      \"related\": [\"system-prompt\", \"llm\"]\n    },\n    {\n      \"id\": \"system-prompt\",\n      \"term\": \"システムプロンプト\",\n      \"aliases\": [\"指示文\", \"instructions\", \"system instruction\"],\n      \"lecture\": 1,\n      \"category\": \"LLM基礎\",\n      \"definition\": \"利用者の質問とは別に、モデルの振る舞い（役割、口調、守るべきルールなど）を指定する入力。Pydantic AIでは Agent の instructions 引数で与える。\",\n      \"related\": [\"prompt\", \"guardrail\"]\n    },\n    {\n      \"id\": \"temperature\",\n      \"term\": \"temperature\",\n      \"aliases\": [\"温度パラメータ\"],\n      \"lecture\": 1,\n      \"category\": \"LLM基礎\",\n      \"definition\": \"次のトークンを選ぶときのランダムさを調整する設定値。低いほど出力が安定し、高いほど多様になる。同じ質問でも回答が変わる理由の一つ。\",\n      \"related\": [\"llm\", \"evaluation-loop\"]\n    },\n    {\n      \"id\": \"hallucination\",\n      \"term\": \"ハルシネーション\",\n      \"aliases\": [\"幻覚\", \"hallucination\"],\n      \"lecture\": 1,\n      \"category\": \"LLM基礎\",\n      \"definition\": \"LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。\",\n      \"related\": [\"tool\", \"evaluation-loop\"]\n    },\n    {\n      \"id\": \"api\",\n      \"term\": \"API\",\n      \"aliases\": [\"Application Programming Interface\", \"Web API\"],\n      \"lecture\": 1,\n      \"category\": \"開発基礎\",\n      \"definition\": \"プログラムから別のサービスの機能を呼び出すための窓口。本講座ではGemini APIをPythonから呼び出してLLMを使う。\",\n      \"related\": [\"api-key\", \"rate-limit\"]\n    },\n    {\n      \"id\": \"api-key\",\n      \"term\": \"APIキー\",\n      \"aliases\": [\"API key\"],\n      \"lecture\": 1,\n      \"category\": \"開発基礎\",\n      \"definition\": \"APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。\",\n      \"related\": [\"codespaces-secret\", \"environment-variable\"]\n    },\n    {\n      \"id\": \"environment-variable\",\n      \"term\": \"環境変数\",\n      \"aliases\": [\"environment variable\", \"env\"],\n      \"lecture\": 1,\n      \"category\": \"開発基礎\",\n      \"definition\": \"プログラムの外側（OSや実行環境）で設定し、プログラムから名前で読み出せる値。APIキーのような秘密情報や、環境ごとに変わる設定をコードから切り離すために使う。\",\n      \"related\": [\"api-key\", \"codespaces-secret\"]\n    },\n    {\n      \"id\": \"rate-limit\",\n      \"term\": \"レート制限\",\n      \"aliases\": [\"rate limit\", \"429エラー\"],\n      \"lecture\": 1,\n      \"category\": \"開発基礎\",\n      \"definition\": \"一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。\",\n      \"related\": [\"api\", \"agent-loop\", \"default-model\"]\n    },\n    {\n      \"id\": \"codespaces\",\n      \"term\": \"GitHub Codespaces\",\n      \"aliases\": [\"Codespaces\", \"コードスペース\"],\n      \"lecture\": 1,\n      \"category\": \"開発環境\",\n      \"definition\": \"GitHubが提供するクラウド上の開発環境。ブラウザからVS Codeを開き、repoに書かれた設定どおりの環境をすぐに使える。個人アカウントには毎月の無料枠があり、使わないときは停止しないと枠を消費し続ける。\",\n      \"related\": [\"devcontainer\", \"codespaces-secret\"]\n    },\n    {\n      \"id\": \"devcontainer\",\n      \"term\": \"devcontainer\",\n      \"aliases\": [\"Dev Container\", \"devcontainer.json\"],\n      \"lecture\": 1,\n      \"category\": \"開発環境\",\n      \"definition\": \"開発環境をコンテナとして定義する仕組み。.devcontainer/devcontainer.json にPythonのバージョンや拡張機能、起動時のコマンドを書いておくと、誰がCodespacesを開いても同じ環境になる。\",\n      \"related\": [\"codespaces\"]\n    },\n    {\n      \"id\": \"codespaces-secret\",\n      \"term\": \"Codespaces secret\",\n      \"aliases\": [\"Codespacesのシークレット\"],\n      \"lecture\": 1,\n      \"category\": \"開発環境\",\n      \"definition\": \"GitHubのアカウント設定に登録しておく秘密の値。指定したrepoのCodespacesを起動すると環境変数として渡される。登録時に使うrepo（自分のfork）を選ぶ必要がある。起動中に登録した場合はCodespacesの再起動が必要。\",\n      \"related\": [\"api-key\", \"environment-variable\", \"codespaces\"]\n    },\n    {\n      \"id\": \"fork\",\n      \"term\": \"fork\",\n      \"aliases\": [\"フォーク\"],\n      \"lecture\": 1,\n      \"category\": \"Git/GitHub\",\n      \"definition\": \"他人のrepoを自分のアカウントに丸ごとコピーすること。本講座では講座repoをforkし、自分のfork上で作業・提出する。\",\n      \"related\": [\"branch\", \"pull-request\"]\n    },\n    {\n      \"id\": \"branch\",\n      \"term\": \"ブランチ\",\n      \"aliases\": [\"branch\"],\n      \"lecture\": 1,\n      \"category\": \"Git/GitHub\",\n      \"definition\": \"作業の流れを分けるための仕組み。main を壊さずに変更を試し、まとまったらPRで main に取り込む。本講座では回ごとに lec01 のような作業ブランチを作る。\",\n      \"related\": [\"pull-request\", \"fork\"]\n    },\n    {\n      \"id\": \"pull-request\",\n      \"term\": \"プルリクエスト（PR）\",\n      \"aliases\": [\"PR\", \"Pull Request\"],\n      \"lecture\": 1,\n      \"category\": \"Git/GitHub\",\n      \"definition\": \"あるブランチの変更を別のブランチへ取り込むよう依頼する仕組み。差分の確認、コメント、自動テストの結果表示ができる。本講座の提出は、自分のfork内で作業ブランチから main へのPRとして行う。\",\n      \"related\": [\"branch\", \"answer-md\"]\n    },\n    {\n      \"id\": \"github-copilot\",\n      \"term\": \"GitHub Copilot\",\n      \"aliases\": [\"Copilot\"],\n      \"lecture\": 1,\n      \"category\": \"AIコーディング\",\n      \"definition\": \"GitHubのAIコーディング支援。エディタでのコード補完、チャットでの質問、エージェントモードでの複数ファイル編集などができる。無料のCopilot Freeは補完の回数とチャットの利用量に上限がある。\",\n      \"related\": [\"code-completion\", \"coding-agent\"]\n    },\n    {\n      \"id\": \"code-completion\",\n      \"term\": \"コード補完\",\n      \"aliases\": [\"インライン補完\", \"inline suggestion\"],\n      \"lecture\": 1,\n      \"category\": \"AIコーディング\",\n      \"definition\": \"書きかけのコードやコメントから、続きのコードをAIが提案する機能。Tabキーなどで受け入れる。提案は正しいとは限らないので、読んで理解してから採用する。\",\n      \"related\": [\"github-copilot\"]\n    },\n    {\n      \"id\": \"coding-agent\",\n      \"term\": \"コーディングエージェント\",\n      \"aliases\": [\"Coding Agent\", \"AIコーディングエージェント\"],\n      \"lecture\": 1,\n      \"category\": \"AIコーディング\",\n      \"definition\": \"ファイルの読み書きやコマンド実行をツールとして使い、指示されたプログラミング作業を複数ステップで自律的に進めるAI。Claude Code、Copilotのエージェントモード、Piなどがある。中身はLLM＋ループ＋ツールというAIエージェントの構造そのもの。\",\n      \"related\": [\"ai-agent\", \"harness\"]\n    },\n    {\n      \"id\": \"pretraining\",\n      \"term\": \"事前学習\",\n      \"aliases\": [\"pre-training\", \"プレトレーニング\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"大量の汎用テキストで、次のトークンを予測するようにモデルを学習させる最初の段階。言語の知識や一般的な常識の多くはここで身につく。\",\n      \"related\": [\"fine-tuning\", \"llm\"]\n    },\n    {\n      \"id\": \"fine-tuning\",\n      \"term\": \"ファインチューニング\",\n      \"aliases\": [\"fine-tuning\", \"追加学習\", \"FT\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"事前学習済みモデルを、特定のタスクや形式に合ったデータで追加学習させること。出力の形式や口調、特定ドメインの振る舞いを身につけさせるのに向く。最新の事実を覚えさせる用途には、ツールやRAGで外部データを参照させるほうが向くことが多い。\",\n      \"related\": [\"sft\", \"peft\", \"lora\"]\n    },\n    {\n      \"id\": \"sft\",\n      \"term\": \"教師ありファインチューニング（SFT）\",\n      \"aliases\": [\"SFT\", \"Supervised Fine-Tuning\", \"指示チューニング\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う振る舞いを身につけさせる代表的な手法。\",\n      \"related\": [\"fine-tuning\", \"lora\"]\n    },\n    {\n      \"id\": \"peft\",\n      \"term\": \"PEFT\",\n      \"aliases\": [\"Parameter-Efficient Fine-Tuning\", \"パラメータ効率の良いファインチューニング\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。\",\n      \"related\": [\"lora\", \"fine-tuning\"]\n    },\n    {\n      \"id\": \"lora\",\n      \"term\": \"LoRA\",\n      \"aliases\": [\"Low-Rank Adaptation\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。\",\n      \"related\": [\"peft\", \"lora-rank\", \"qlora\"]\n    },\n    {\n      \"id\": \"lora-rank\",\n      \"term\": \"ランク（LoRAのr）\",\n      \"aliases\": [\"rank\", \"r\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"LoRAで使う低ランク行列の大きさを決めるハイパーパラメータ。大きいほど表現力が上がるが、学習するパラメータとメモリも増える。\",\n      \"related\": [\"lora\"]\n    },\n    {\n      \"id\": \"qlora\",\n      \"term\": \"QLoRA\",\n      \"aliases\": [\"Quantized LoRA\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。\",\n      \"related\": [\"lora\", \"quantization\"]\n    },\n    {\n      \"id\": \"quantization\",\n      \"term\": \"量子化\",\n      \"aliases\": [\"quantization\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。\",\n      \"related\": [\"qlora\"]\n    },\n    {\n      \"id\": \"learning-rate\",\n      \"term\": \"学習率\",\n      \"aliases\": [\"learning rate\", \"lr\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。\",\n      \"related\": [\"epoch\", \"overfitting\"]\n    },\n    {\n      \"id\": \"epoch\",\n      \"term\": \"エポック\",\n      \"aliases\": [\"epoch\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。\",\n      \"related\": [\"overfitting\", \"learning-rate\"]\n    },\n    {\n      \"id\": \"overfitting\",\n      \"term\": \"過学習\",\n      \"aliases\": [\"overfitting\", \"オーバーフィッティング\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。\",\n      \"related\": [\"epoch\", \"evaluation-loop\"]\n    },\n    {\n      \"id\": \"huggingface-hub\",\n      \"term\": \"Hugging Face Hub\",\n      \"aliases\": [\"Hugging Face\", \"HF Hub\"],\n      \"lecture\": 2,\n      \"category\": \"ファインチューニング\",\n      \"definition\": \"公開されたモデルやデータセットを共有・ダウンロードできるプラットフォーム。オープンウェイトのモデルを入手したり、LoRAアダプタを公開したりするのに使われる。\",\n      \"related\": [\"lora\", \"pretraining\"]\n    },\n    {\n      \"id\": \"ai-agent\",\n      \"term\": \"AIエージェント\",\n      \"aliases\": [\"エージェント\", \"agent\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"LLMが自分で判断してツールを呼び出し、その結果を見てさらに次の行動を決める、というループで目的を達成するシステム。基本構造は「モデル＋ループ＋ツール」。\",\n      \"related\": [\"agent-loop\", \"tool\", \"harness\"]\n    },\n    {\n      \"id\": \"agent-loop\",\n      \"term\": \"エージェントループ\",\n      \"aliases\": [\"agent loop\", \"ツール呼び出しループ\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"モデルに入力を渡す→モデルがツール呼び出しを返したら実行する→結果をモデルに戻す、をモデルが最終回答を返すまで繰り返す処理。1つの質問でLLMを複数回呼ぶため、時間・料金・レート制限への影響が大きい。\",\n      \"related\": [\"ai-agent\", \"tool\", \"rate-limit\"]\n    },\n    {\n      \"id\": \"tool\",\n      \"term\": \"ツール（Tool use / Function calling）\",\n      \"aliases\": [\"ツール呼び出し\", \"Function calling\", \"Tool use\", \"ツール\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"モデルが呼び出せるように用意した関数。モデルはツールの名前・説明・引数の型を見て、必要なときに引数を決めて呼び出しを依頼し、実行はプログラム側が行う。検索、計算、ファイル操作、外部APIの呼び出しなどに使う。\",\n      \"related\": [\"tool-description\", \"agent-loop\"]\n    },\n    {\n      \"id\": \"tool-description\",\n      \"term\": \"ツールの説明文\",\n      \"aliases\": [\"docstring\", \"ツールスキーマ\", \"tool schema\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"ツールの用途や引数の意味をモデルに伝える文章。Pydantic AIでは関数のdocstringと型ヒントから自動で作られる。モデルはこれを読んでツールを使うか、どの引数で呼ぶかを決めるので、書き方次第で挙動が大きく変わる。\",\n      \"related\": [\"tool\", \"pydantic-ai\"]\n    },\n    {\n      \"id\": \"structured-output\",\n      \"term\": \"構造化出力\",\n      \"aliases\": [\"structured output\", \"JSON出力\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"モデルの出力を、決めた型やJSONスキーマに沿った形で受け取る仕組み。プログラムで後処理しやすくなる。Pydantic AIでは output_type にPydanticモデルを指定する。\",\n      \"related\": [\"pydantic-ai\"]\n    },\n    {\n      \"id\": \"harness\",\n      \"term\": \"ハーネス\",\n      \"aliases\": [\"agent harness\", \"エージェントハーネス\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"LLMの周りを、ループ・ツール・画面（UI）などで包み、実際に作業できるようにするプログラム。Claude CodeやPiはハーネスの例。Piは組み込みツールをread・write・edit・bashの4つに絞った薄いハーネスとして知られる。\",\n      \"related\": [\"coding-agent\", \"pi\"]\n    },\n    {\n      \"id\": \"pi\",\n      \"term\": \"Pi\",\n      \"aliases\": [\"Pi coding agent\", \"pi agent\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"ターミナルで動くオープンソース（MITライセンス）のコーディングエージェント。最小限の機能とシステムプロンプトで構成し、利用者が拡張して使う設計。2026年10月に1.0が公開された。npmのパッケージ名は @earendil-works/pi-coding-agent。\",\n      \"related\": [\"harness\", \"coding-agent\"]\n    },\n    {\n      \"id\": \"pydantic-ai\",\n      \"term\": \"Pydantic AI\",\n      \"aliases\": [\"pydantic-ai\", \"PydanticAI\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"Pythonでエージェントを作るためのフレームワーク。Pydanticの型を使ってツールの引数や出力を定義でき、GeminiやOpenAIなど多くのモデルを同じ書き方で扱える。テスト用のモデルでAPIを呼ばずに動作確認できる。本講座第3回の本線。\",\n      \"related\": [\"test-model\", \"structured-output\", \"adk\"]\n    },\n    {\n      \"id\": \"adk\",\n      \"term\": \"ADK（Agent Development Kit）\",\n      \"aliases\": [\"ADK\", \"Agent Development Kit\", \"google-adk\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"Googleが公開しているオープンソースのエージェント開発フレームワーク。Geminiとの統合が深く、adk web でツール呼び出しの流れを画面で確認できる。2.0で1.x系から破壊的変更が入っている。本講座では宿題の発展レベルで扱う。\",\n      \"related\": [\"pydantic-ai\", \"ai-agent\"]\n    },\n    {\n      \"id\": \"mcp\",\n      \"term\": \"MCP（Model Context Protocol）\",\n      \"aliases\": [\"MCP\", \"Model Context Protocol\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。\",\n      \"related\": [\"tool\", \"ai-agent\", \"mcp-server\", \"mcp-client\"]\n    },\n    {\n      \"id\": \"guardrail\",\n      \"term\": \"ガードレール\",\n      \"aliases\": [\"guardrail\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"エージェントが望ましくない行動や出力をしないように設ける制約や検査。指示文でのルール、ツールの権限を絞る、出力をチェックする、重要な操作の前に人の確認を挟む、などの方法がある。\",\n      \"related\": [\"human-in-the-loop\", \"system-prompt\"]\n    },\n    {\n      \"id\": \"human-in-the-loop\",\n      \"term\": \"Human-in-the-loop\",\n      \"aliases\": [\"HITL\", \"人間による承認\", \"確認を挟む\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"エージェントの処理の途中に人の確認や承認を組み込むこと。データの書き込みや送信など、取り消しにくい操作の前に使う。Pydantic AIでは requires_approval=True を指定したツールで実現できる。\",\n      \"related\": [\"guardrail\", \"tool\"]\n    },\n    {\n      \"id\": \"test-model\",\n      \"term\": \"TestModel\",\n      \"aliases\": [\"テスト用モデル\"],\n      \"lecture\": 3,\n      \"category\": \"エージェント\",\n      \"definition\": \"Pydantic AIに含まれるテスト用の偽モデル。実際のLLMを呼ばずに、ツールを呼び出してダミーの回答を返す。APIキーや利用枠を使わずに、エージェントの配線（ツールが登録され、呼ばれ、結果が返るか）をテストできる。\",\n      \"related\": [\"pydantic-ai\", \"rate-limit\"]\n    },\n    {\n      \"id\": \"homework-levels\",\n      \"term\": \"宿題の3レベル\",\n      \"aliases\": [\"3レベル\", \"宿題レベル\"],\n      \"lecture\": 1,\n      \"category\": \"講座\",\n      \"definition\": \"AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。\",\n      \"related\": [\"answer-md\"]\n    },\n    {\n      \"id\": \"answer-md\",\n      \"term\": \"answer.md\",\n      \"aliases\": [\"解答ファイル\"],\n      \"lecture\": 1,\n      \"category\": \"講座\",\n      \"definition\": \"AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。\",\n      \"related\": [\"pull-request\", \"homework-levels\"]\n    },\n    {\n      \"id\": \"lec-start-tag\",\n      \"term\": \"lecNN-start タグ\",\n      \"aliases\": [\"開始タグ\", \"lec03-start\"],\n      \"lecture\": 1,\n      \"category\": \"講座\",\n      \"definition\": \"AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。\",\n      \"related\": [\"branch\"]\n    },\n    {\n      \"id\": \"default-model\",\n      \"term\": \"既定モデル\",\n      \"aliases\": [\"デフォルトモデル\", \"gemini-3.5-flash-lite\"],\n      \"lecture\": 1,\n      \"category\": \"講座\",\n      \"definition\": \"AIE2026の演習と宿題で標準に使うモデルは gemini-3.5-flash-lite。無料枠のレート制限と講座運営の安定性を考えて選んでいる。環境変数 AIE_MODEL で他のモデルに切り替えてよく、自分で契約したAPIを使ってもよい。\",\n      \"related\": [\"rate-limit\", \"api-key\"]\n    },\n    {\n      \"id\": \"external-tool\",\n      \"term\": \"外部ツール\",\n      \"aliases\": [\"external tool\", \"外部サービス\"],\n      \"lecture\": 5,\n      \"category\": \"MCP\",\n      \"definition\": \"Webページの取得、ファイル操作、データベース、他社のAPIなど、エージェントのプログラムの外にある機能をツールとして使うこと。第3回の関数ツールは同じプロセスの中で動くが、外部ツールはプロセスや機械をまたいで呼び出す。MCPはそのための共通の規約。\",\n      \"related\": [\"tool\", \"mcp\", \"mcp-server\"]\n    },\n    {\n      \"id\": \"mcp-server\",\n      \"term\": \"MCPサーバー\",\n      \"aliases\": [\"MCP server\"],\n      \"lecture\": 5,\n      \"category\": \"MCP\",\n      \"definition\": \"MCPでツール（やリソース、プロンプト）を公開する側のプログラム。公開するのはツールの名前・説明文・引数のスキーマで、クライアントの要求に応じてツールを実行し結果を返す。本講座では用語集の検索をMCPサーバーにする（app/mcp_server.py）。公開されている既製のサーバー（fetch、filesystem、GitHubなど）も多い。\",\n      \"related\": [\"mcp\", \"mcp-client\", \"mcp-transport\", \"fetch-server\"]\n    },\n    {\n      \"id\": \"mcp-client\",\n      \"term\": \"MCPクライアント\",\n      \"aliases\": [\"MCP client\", \"MCPホスト\"],\n      \"lecture\": 5,\n      \"category\": \"MCP\",\n      \"definition\": \"MCPサーバーに接続し、公開されたツールをモデルに渡して、モデルの指示でツールを呼び出す側。エージェントのプログラム（Pydantic AIの MCPToolset）や、Copilot Chat・Claude Desktop・Cursorなどのエディタやアプリがクライアントになる。同じサーバーをどのクライアントからも使えるのがMCPの利点。\",\n      \"related\": [\"mcp\", \"mcp-server\", \"mcp-config\"]\n    },\n    {\n      \"id\": \"mcp-transport\",\n      \"term\": \"トランスポート（stdio・Streamable HTTP）\",\n      \"aliases\": [\"stdio\", \"Streamable HTTP\", \"transport\", \"MCPのトランスポート\"],\n      \"lecture\": 5,\n      \"category\": \"MCP\",\n      \"definition\": \"MCPのクライアントとサーバーがやりとりする経路。stdioでは、クライアントがサーバーをサブプロセスとして起動し、標準入出力でメッセージを交換する（手軽で、同じ機械の中で使う）。Streamable HTTPでは、サーバーがURLで待ち受け、別のプロセスや別の機械から接続できる（公開範囲と認証を考える必要がある）。\",\n      \"related\": [\"mcp-server\", \"mcp-client\"]\n    },\n    {\n      \"id\": \"mcp-config\",\n      \"term\": \"mcpServers設定\",\n      \"aliases\": [\"mcpServers\", \"MCPの設定ファイル\", \"mcp_servers.json\", \"mcp.json\"],\n      \"lecture\": 5,\n      \"category\": \"MCP\",\n      \"definition\": \"使うMCPサーバーを列挙する設定ファイル。サーバーごとに名前と、起動コマンド（stdio）または接続先URL（HTTP）を書く。Claude Desktop、Cursor、Pydantic AIなどが同じ mcpServers 形式を読む（VS Code / Copilot Chatは servers 形式）。本講座では app/mcp_servers.json。設定を数行足すだけでツールが増えるのがMCPの体験の核心。\",\n      \"related\": [\"mcp-client\", \"tool-namespace\"]\n    },\n    {\n      \"id\": \"tool-namespace\",\n      \"term\": \"ツールの名前空間\",\n      \"aliases\": [\"ツール名のプレフィックス\", \"prefix\", \"名前の衝突\"],\n      \"lecture\": 5,\n      \"category\": \"MCP\",\n      \"definition\": \"複数のMCPサーバーを使うとき、同じ名前のツールを区別するために「サーバー名_ツール名」のように接頭辞を付けること。本講座の app/agent.py は設定ファイルのサーバー名を自動で付ける（例: glossary_search_glossary。Pydantic AIの load_mcp_toolsets も同じ）。ツールが増えるほどモデルに渡す説明文も増え、呼び分けの精度とトークン数に影響する。\",\n      \"related\": [\"mcp-config\", \"tool-description\", \"rate-limit\"]\n    },\n    {\n      \"id\": \"fetch-server\",\n      \"term\": \"fetchサーバー\",\n      \"aliases\": [\"mcp-server-fetch\", \"fetch\", \"Web取得ツール\"],\n      \"lecture\": 5,\n      \"category\": \"MCP\",\n      \"definition\": \"指定したURLのページを取得し、本文をMarkdownにして返す既製のMCPサーバー（Python製、uvx mcp-server-fetch で起動）。本講座では「設定ファイルに書くだけで外部ツールが増える」例として使う。モデルが選んだURLに実際にアクセスするので、どこへ何を送るかは利用者が把握しておく。\",\n      \"related\": [\"mcp-server\", \"external-tool\", \"guardrail\"]\n    }\n  ]\n}\n",
  "glossary_sft.jsonl": "{\"messages\": [{\"role\": \"user\", \"content\": \"「AIエンジニアリング」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"AIエンジニアリングについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「評価と改善のサイクル」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"作る→測る→直すを繰り返す開発の進め方。AIアプリは出力が確率的に揺れるため、感覚ではなく評価データや指標で良し悪しを判断し、その結果をもとにプロンプト・データ・モデル・ツールを改善する。本講座では学習支援エージェントを題材に、回をまたいでこのサイクルを回す。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"評価と改善のサイクルについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"作る→測る→直すを繰り返す開発の進め方。AIアプリは出力が確率的に揺れるため、感覚ではなく評価データや指標で良し悪しを判断し、その結果をもとにプロンプト・データ・モデル・ツールを改善する。本講座では学習支援エージェントを題材に、回をまたいでこのサイクルを回す。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「LLM（大規模言語モデル）」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"大量のテキストで学習し、入力に続くトークンを予測することで文章を生成するモデル。質問応答、要約、コード生成など多くのタスクを、指示（プロンプト）を変えるだけでこなせる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"LLM（大規模言語モデル）について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"大量のテキストで学習し、入力に続くトークンを予測することで文章を生成するモデル。質問応答、要約、コード生成など多くのタスクを、指示（プロンプト）を変えるだけでこなせる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「トークン」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"トークンについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「コンテキストウィンドウ」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"コンテキストウィンドウについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「プロンプト」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"LLMに渡す入力文。質問だけでなく、役割、制約、出力形式、例などを含めることで出力を方向づけられる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"プロンプトについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"LLMに渡す入力文。質問だけでなく、役割、制約、出力形式、例などを含めることで出力を方向づけられる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「システムプロンプト」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"利用者の質問とは別に、モデルの振る舞い（役割、口調、守るべきルールなど）を指定する入力。Pydantic AIでは Agent の instructions 引数で与える。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"システムプロンプトについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"利用者の質問とは別に、モデルの振る舞い（役割、口調、守るべきルールなど）を指定する入力。Pydantic AIでは Agent の instructions 引数で与える。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「temperature」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"次のトークンを選ぶときのランダムさを調整する設定値。低いほど出力が安定し、高いほど多様になる。同じ質問でも回答が変わる理由の一つ。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"temperatureについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"次のトークンを選ぶときのランダムさを調整する設定値。低いほど出力が安定し、高いほど多様になる。同じ質問でも回答が変わる理由の一つ。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「ハルシネーション」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"ハルシネーションについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「API」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"プログラムから別のサービスの機能を呼び出すための窓口。本講座ではGemini APIをPythonから呼び出してLLMを使う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"APIについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"プログラムから別のサービスの機能を呼び出すための窓口。本講座ではGemini APIをPythonから呼び出してLLMを使う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「APIキー」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"APIキーについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「環境変数」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"プログラムの外側（OSや実行環境）で設定し、プログラムから名前で読み出せる値。APIキーのような秘密情報や、環境ごとに変わる設定をコードから切り離すために使う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"環境変数について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"プログラムの外側（OSや実行環境）で設定し、プログラムから名前で読み出せる値。APIキーのような秘密情報や、環境ごとに変わる設定をコードから切り離すために使う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「レート制限」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"レート制限について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「GitHub Codespaces」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"GitHubが提供するクラウド上の開発環境。ブラウザからVS Codeを開き、repoに書かれた設定どおりの環境をすぐに使える。個人アカウントには毎月の無料枠があり、使わないときは停止しないと枠を消費し続ける。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"GitHub Codespacesについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"GitHubが提供するクラウド上の開発環境。ブラウザからVS Codeを開き、repoに書かれた設定どおりの環境をすぐに使える。個人アカウントには毎月の無料枠があり、使わないときは停止しないと枠を消費し続ける。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「devcontainer」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"開発環境をコンテナとして定義する仕組み。.devcontainer/devcontainer.json にPythonのバージョンや拡張機能、起動時のコマンドを書いておくと、誰がCodespacesを開いても同じ環境になる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"devcontainerについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"開発環境をコンテナとして定義する仕組み。.devcontainer/devcontainer.json にPythonのバージョンや拡張機能、起動時のコマンドを書いておくと、誰がCodespacesを開いても同じ環境になる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「Codespaces secret」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"GitHubのアカウント設定に登録しておく秘密の値。指定したrepoのCodespacesを起動すると環境変数として渡される。登録時に使うrepo（自分のfork）を選ぶ必要がある。起動中に登録した場合はCodespacesの再起動が必要。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"Codespaces secretについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"GitHubのアカウント設定に登録しておく秘密の値。指定したrepoのCodespacesを起動すると環境変数として渡される。登録時に使うrepo（自分のfork）を選ぶ必要がある。起動中に登録した場合はCodespacesの再起動が必要。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「fork」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"他人のrepoを自分のアカウントに丸ごとコピーすること。本講座では講座repoをforkし、自分のfork上で作業・提出する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"forkについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"他人のrepoを自分のアカウントに丸ごとコピーすること。本講座では講座repoをforkし、自分のfork上で作業・提出する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「ブランチ」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"作業の流れを分けるための仕組み。main を壊さずに変更を試し、まとまったらPRで main に取り込む。本講座では回ごとに lec01 のような作業ブランチを作る。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"ブランチについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"作業の流れを分けるための仕組み。main を壊さずに変更を試し、まとまったらPRで main に取り込む。本講座では回ごとに lec01 のような作業ブランチを作る。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「プルリクエスト（PR）」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"あるブランチの変更を別のブランチへ取り込むよう依頼する仕組み。差分の確認、コメント、自動テストの結果表示ができる。本講座の提出は、自分のfork内で作業ブランチから main へのPRとして行う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"プルリクエスト（PR）について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"あるブランチの変更を別のブランチへ取り込むよう依頼する仕組み。差分の確認、コメント、自動テストの結果表示ができる。本講座の提出は、自分のfork内で作業ブランチから main へのPRとして行う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「GitHub Copilot」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"GitHubのAIコーディング支援。エディタでのコード補完、チャットでの質問、エージェントモードでの複数ファイル編集などができる。無料のCopilot Freeは補完の回数とチャットの利用量に上限がある。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"GitHub Copilotについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"GitHubのAIコーディング支援。エディタでのコード補完、チャットでの質問、エージェントモードでの複数ファイル編集などができる。無料のCopilot Freeは補完の回数とチャットの利用量に上限がある。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「コード補完」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"書きかけのコードやコメントから、続きのコードをAIが提案する機能。Tabキーなどで受け入れる。提案は正しいとは限らないので、読んで理解してから採用する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"コード補完について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"書きかけのコードやコメントから、続きのコードをAIが提案する機能。Tabキーなどで受け入れる。提案は正しいとは限らないので、読んで理解してから採用する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「コーディングエージェント」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"ファイルの読み書きやコマンド実行をツールとして使い、指示されたプログラミング作業を複数ステップで自律的に進めるAI。Claude Code、Copilotのエージェントモード、Piなどがある。中身はLLM＋ループ＋ツールというAIエージェントの構造そのもの。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"コーディングエージェントについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"ファイルの読み書きやコマンド実行をツールとして使い、指示されたプログラミング作業を複数ステップで自律的に進めるAI。Claude Code、Copilotのエージェントモード、Piなどがある。中身はLLM＋ループ＋ツールというAIエージェントの構造そのもの。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「事前学習」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"大量の汎用テキストで、次のトークンを予測するようにモデルを学習させる最初の段階。言語の知識や一般的な常識の多くはここで身につく。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"事前学習について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"大量の汎用テキストで、次のトークンを予測するようにモデルを学習させる最初の段階。言語の知識や一般的な常識の多くはここで身につく。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「ファインチューニング」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"事前学習済みモデルを、特定のタスクや形式に合ったデータで追加学習させること。出力の形式や口調、特定ドメインの振る舞いを身につけさせるのに向く。最新の事実を覚えさせる用途には、ツールやRAGで外部データを参照させるほうが向くことが多い。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"ファインチューニングについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"事前学習済みモデルを、特定のタスクや形式に合ったデータで追加学習させること。出力の形式や口調、特定ドメインの振る舞いを身につけさせるのに向く。最新の事実を覚えさせる用途には、ツールやRAGで外部データを参照させるほうが向くことが多い。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「教師ありファインチューニング（SFT）」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う振る舞いを身につけさせる代表的な手法。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"教師ありファインチューニング（SFT）について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う振る舞いを身につけさせる代表的な手法。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「PEFT」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"PEFTについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「LoRA」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"LoRAについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「ランク（LoRAのr）」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"LoRAで使う低ランク行列の大きさを決めるハイパーパラメータ。大きいほど表現力が上がるが、学習するパラメータとメモリも増える。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"ランク（LoRAのr）について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"LoRAで使う低ランク行列の大きさを決めるハイパーパラメータ。大きいほど表現力が上がるが、学習するパラメータとメモリも増える。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「QLoRA」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"QLoRAについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「量子化」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"量子化について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「学習率」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"学習率について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「エポック」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"エポックについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「過学習」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"過学習について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「Hugging Face Hub」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"公開されたモデルやデータセットを共有・ダウンロードできるプラットフォーム。オープンウェイトのモデルを入手したり、LoRAアダプタを公開したりするのに使われる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"Hugging Face Hubについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"公開されたモデルやデータセットを共有・ダウンロードできるプラットフォーム。オープンウェイトのモデルを入手したり、LoRAアダプタを公開したりするのに使われる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「AIエージェント」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"LLMが自分で判断してツールを呼び出し、その結果を見てさらに次の行動を決める、というループで目的を達成するシステム。基本構造は「モデル＋ループ＋ツール」。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"AIエージェントについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"LLMが自分で判断してツールを呼び出し、その結果を見てさらに次の行動を決める、というループで目的を達成するシステム。基本構造は「モデル＋ループ＋ツール」。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「エージェントループ」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"モデルに入力を渡す→モデルがツール呼び出しを返したら実行する→結果をモデルに戻す、をモデルが最終回答を返すまで繰り返す処理。1つの質問でLLMを複数回呼ぶため、時間・料金・レート制限への影響が大きい。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"エージェントループについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"モデルに入力を渡す→モデルがツール呼び出しを返したら実行する→結果をモデルに戻す、をモデルが最終回答を返すまで繰り返す処理。1つの質問でLLMを複数回呼ぶため、時間・料金・レート制限への影響が大きい。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「ツール（Tool use / Function calling）」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"モデルが呼び出せるように用意した関数。モデルはツールの名前・説明・引数の型を見て、必要なときに引数を決めて呼び出しを依頼し、実行はプログラム側が行う。検索、計算、ファイル操作、外部APIの呼び出しなどに使う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"ツール（Tool use / Function calling）について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"モデルが呼び出せるように用意した関数。モデルはツールの名前・説明・引数の型を見て、必要なときに引数を決めて呼び出しを依頼し、実行はプログラム側が行う。検索、計算、ファイル操作、外部APIの呼び出しなどに使う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「ツールの説明文」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"ツールの用途や引数の意味をモデルに伝える文章。Pydantic AIでは関数のdocstringと型ヒントから自動で作られる。モデルはこれを読んでツールを使うか、どの引数で呼ぶかを決めるので、書き方次第で挙動が大きく変わる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"ツールの説明文について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"ツールの用途や引数の意味をモデルに伝える文章。Pydantic AIでは関数のdocstringと型ヒントから自動で作られる。モデルはこれを読んでツールを使うか、どの引数で呼ぶかを決めるので、書き方次第で挙動が大きく変わる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「構造化出力」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"モデルの出力を、決めた型やJSONスキーマに沿った形で受け取る仕組み。プログラムで後処理しやすくなる。Pydantic AIでは output_type にPydanticモデルを指定する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"構造化出力について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"モデルの出力を、決めた型やJSONスキーマに沿った形で受け取る仕組み。プログラムで後処理しやすくなる。Pydantic AIでは output_type にPydanticモデルを指定する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「ハーネス」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"LLMの周りを、ループ・ツール・画面（UI）などで包み、実際に作業できるようにするプログラム。Claude CodeやPiはハーネスの例。Piは組み込みツールをread・write・edit・bashの4つに絞った薄いハーネスとして知られる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"ハーネスについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"LLMの周りを、ループ・ツール・画面（UI）などで包み、実際に作業できるようにするプログラム。Claude CodeやPiはハーネスの例。Piは組み込みツールをread・write・edit・bashの4つに絞った薄いハーネスとして知られる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「Pi」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"ターミナルで動くオープンソース（MITライセンス）のコーディングエージェント。最小限の機能とシステムプロンプトで構成し、利用者が拡張して使う設計。2026年10月に1.0が公開された。npmのパッケージ名は @earendil-works/pi-coding-agent。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"Piについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"ターミナルで動くオープンソース（MITライセンス）のコーディングエージェント。最小限の機能とシステムプロンプトで構成し、利用者が拡張して使う設計。2026年10月に1.0が公開された。npmのパッケージ名は @earendil-works/pi-coding-agent。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「Pydantic AI」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"Pythonでエージェントを作るためのフレームワーク。Pydanticの型を使ってツールの引数や出力を定義でき、GeminiやOpenAIなど多くのモデルを同じ書き方で扱える。テスト用のモデルでAPIを呼ばずに動作確認できる。本講座第3回の本線。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"Pydantic AIについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"Pythonでエージェントを作るためのフレームワーク。Pydanticの型を使ってツールの引数や出力を定義でき、GeminiやOpenAIなど多くのモデルを同じ書き方で扱える。テスト用のモデルでAPIを呼ばずに動作確認できる。本講座第3回の本線。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「ADK（Agent Development Kit）」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"Googleが公開しているオープンソースのエージェント開発フレームワーク。Geminiとの統合が深く、adk web でツール呼び出しの流れを画面で確認できる。2.0で1.x系から破壊的変更が入っている。本講座では宿題の発展レベルで扱う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"ADK（Agent Development Kit）について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"Googleが公開しているオープンソースのエージェント開発フレームワーク。Geminiとの統合が深く、adk web でツール呼び出しの流れを画面で確認できる。2.0で1.x系から破壊的変更が入っている。本講座では宿題の発展レベルで扱う。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「MCP（Model Context Protocol）」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"MCP（Model Context Protocol）について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「ガードレール」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"エージェントが望ましくない行動や出力をしないように設ける制約や検査。指示文でのルール、ツールの権限を絞る、出力をチェックする、重要な操作の前に人の確認を挟む、などの方法がある。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"ガードレールについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"エージェントが望ましくない行動や出力をしないように設ける制約や検査。指示文でのルール、ツールの権限を絞る、出力をチェックする、重要な操作の前に人の確認を挟む、などの方法がある。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「Human-in-the-loop」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"エージェントの処理の途中に人の確認や承認を組み込むこと。データの書き込みや送信など、取り消しにくい操作の前に使う。Pydantic AIでは requires_approval=True を指定したツールで実現できる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"Human-in-the-loopについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"エージェントの処理の途中に人の確認や承認を組み込むこと。データの書き込みや送信など、取り消しにくい操作の前に使う。Pydantic AIでは requires_approval=True を指定したツールで実現できる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「TestModel」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"Pydantic AIに含まれるテスト用の偽モデル。実際のLLMを呼ばずに、ツールを呼び出してダミーの回答を返す。APIキーや利用枠を使わずに、エージェントの配線（ツールが登録され、呼ばれ、結果が返るか）をテストできる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"TestModelについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"Pydantic AIに含まれるテスト用の偽モデル。実際のLLMを呼ばずに、ツールを呼び出してダミーの回答を返す。APIキーや利用枠を使わずに、エージェントの配線（ツールが登録され、呼ばれ、結果が返るか）をテストできる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「宿題の3レベル」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"宿題の3レベルについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「answer.md」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"answer.mdについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「lecNN-start タグ」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"lecNN-start タグについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「既定モデル」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"AIE2026の演習と宿題で標準に使うモデルは gemini-3.5-flash-lite。無料枠のレート制限と講座運営の安定性を考えて選んでいる。環境変数 AIE_MODEL で他のモデルに切り替えてよく、自分で契約したAPIを使ってもよい。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"既定モデルについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"AIE2026の演習と宿題で標準に使うモデルは gemini-3.5-flash-lite。無料枠のレート制限と講座運営の安定性を考えて選んでいる。環境変数 AIE_MODEL で他のモデルに切り替えてよく、自分で契約したAPIを使ってもよい。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「外部ツール」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"Webページの取得、ファイル操作、データベース、他社のAPIなど、エージェントのプログラムの外にある機能をツールとして使うこと。第3回の関数ツールは同じプロセスの中で動くが、外部ツールはプロセスや機械をまたいで呼び出す。MCPはそのための共通の規約。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"外部ツールについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"Webページの取得、ファイル操作、データベース、他社のAPIなど、エージェントのプログラムの外にある機能をツールとして使うこと。第3回の関数ツールは同じプロセスの中で動くが、外部ツールはプロセスや機械をまたいで呼び出す。MCPはそのための共通の規約。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「MCPサーバー」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"MCPでツール（やリソース、プロンプト）を公開する側のプログラム。公開するのはツールの名前・説明文・引数のスキーマで、クライアントの要求に応じてツールを実行し結果を返す。本講座では用語集の検索をMCPサーバーにする（app/mcp_server.py）。公開されている既製のサーバー（fetch、filesystem、GitHubなど）も多い。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"MCPサーバーについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"MCPでツール（やリソース、プロンプト）を公開する側のプログラム。公開するのはツールの名前・説明文・引数のスキーマで、クライアントの要求に応じてツールを実行し結果を返す。本講座では用語集の検索をMCPサーバーにする（app/mcp_server.py）。公開されている既製のサーバー（fetch、filesystem、GitHubなど）も多い。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「MCPクライアント」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"MCPサーバーに接続し、公開されたツールをモデルに渡して、モデルの指示でツールを呼び出す側。エージェントのプログラム（Pydantic AIの MCPToolset）や、Copilot Chat・Claude Desktop・Cursorなどのエディタやアプリがクライアントになる。同じサーバーをどのクライアントからも使えるのがMCPの利点。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"MCPクライアントについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"MCPサーバーに接続し、公開されたツールをモデルに渡して、モデルの指示でツールを呼び出す側。エージェントのプログラム（Pydantic AIの MCPToolset）や、Copilot Chat・Claude Desktop・Cursorなどのエディタやアプリがクライアントになる。同じサーバーをどのクライアントからも使えるのがMCPの利点。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「トランスポート（stdio・Streamable HTTP）」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"MCPのクライアントとサーバーがやりとりする経路。stdioでは、クライアントがサーバーをサブプロセスとして起動し、標準入出力でメッセージを交換する（手軽で、同じ機械の中で使う）。Streamable HTTPでは、サーバーがURLで待ち受け、別のプロセスや別の機械から接続できる（公開範囲と認証を考える必要がある）。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"トランスポート（stdio・Streamable HTTP）について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"MCPのクライアントとサーバーがやりとりする経路。stdioでは、クライアントがサーバーをサブプロセスとして起動し、標準入出力でメッセージを交換する（手軽で、同じ機械の中で使う）。Streamable HTTPでは、サーバーがURLで待ち受け、別のプロセスや別の機械から接続できる（公開範囲と認証を考える必要がある）。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「mcpServers設定」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"使うMCPサーバーを列挙する設定ファイル。サーバーごとに名前と、起動コマンド（stdio）または接続先URL（HTTP）を書く。Claude Desktop、Cursor、Pydantic AIなどが同じ mcpServers 形式を読む（VS Code / Copilot Chatは servers 形式）。本講座では app/mcp_servers.json。設定を数行足すだけでツールが増えるのがMCPの体験の核心。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"mcpServers設定について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"使うMCPサーバーを列挙する設定ファイル。サーバーごとに名前と、起動コマンド（stdio）または接続先URL（HTTP）を書く。Claude Desktop、Cursor、Pydantic AIなどが同じ mcpServers 形式を読む（VS Code / Copilot Chatは servers 形式）。本講座では app/mcp_servers.json。設定を数行足すだけでツールが増えるのがMCPの体験の核心。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「ツールの名前空間」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"複数のMCPサーバーを使うとき、同じ名前のツールを区別するために「サーバー名_ツール名」のように接頭辞を付けること。本講座の app/agent.py は設定ファイルのサーバー名を自動で付ける（例: glossary_search_glossary。Pydantic AIの load_mcp_toolsets も同じ）。ツールが増えるほどモデルに渡す説明文も増え、呼び分けの精度とトークン数に影響する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"ツールの名前空間について簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"複数のMCPサーバーを使うとき、同じ名前のツールを区別するために「サーバー名_ツール名」のように接頭辞を付けること。本講座の app/agent.py は設定ファイルのサーバー名を自動で付ける（例: glossary_search_glossary。Pydantic AIの load_mcp_toolsets も同じ）。ツールが増えるほどモデルに渡す説明文も増え、呼び分けの精度とトークン数に影響する。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"「fetchサーバー」とは何ですか？\"}, {\"role\": \"assistant\", \"content\": \"指定したURLのページを取得し、本文をMarkdownにして返す既製のMCPサーバー（Python製、uvx mcp-server-fetch で起動）。本講座では「設定ファイルに書くだけで外部ツールが増える」例として使う。モデルが選んだURLに実際にアクセスするので、どこへ何を送るかは利用者が把握しておく。\"}]}\n{\"messages\": [{\"role\": \"user\", \"content\": \"fetchサーバーについて簡単に説明してください。\"}, {\"role\": \"assistant\", \"content\": \"指定したURLのページを取得し、本文をMarkdownにして返す既製のMCPサーバー（Python製、uvx mcp-server-fetch で起動）。本講座では「設定ファイルに書くだけで外部ツールが増える」例として使う。モデルが選んだURLに実際にアクセスするので、どこへ何を送るかは利用者が把握しておく。\"}]}\n",
  "evaluation.json": "[\n  {\n    \"id\": \"lora_seen\",\n    \"category\": \"seen\",\n    \"domain\": \"general\",\n    \"question\": \"「LoRA」とは何ですか？\",\n    \"query\": \"LoRA\",\n    \"expected_term_id\": \"lora\",\n    \"reference_answer\": \"事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。\",\n    \"keyword_groups\": [\n      [\n        \"固定\",\n        \"更新しない\",\n        \"凍結\"\n      ],\n      [\n        \"低ランク\",\n        \"小さな行列\",\n        \"低いランク\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"lora_paraphrase\",\n    \"category\": \"paraphrase\",\n    \"domain\": \"general\",\n    \"question\": \"LoRAでは元の重みをどう扱い、何を学習しますか？\",\n    \"query\": \"LoRA\",\n    \"expected_term_id\": \"lora\",\n    \"reference_answer\": \"事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。\",\n    \"keyword_groups\": [\n      [\n        \"固定\",\n        \"更新しない\",\n        \"凍結\"\n      ],\n      [\n        \"低ランク\",\n        \"小さな行列\",\n        \"低いランク\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"qlora_seen\",\n    \"category\": \"seen\",\n    \"domain\": \"general\",\n    \"question\": \"「QLoRA」とは何ですか？\",\n    \"query\": \"QLoRA\",\n    \"expected_term_id\": \"qlora\",\n    \"reference_answer\": \"ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。\",\n    \"keyword_groups\": [\n      [\n        \"量子化\"\n      ],\n      [\n        \"4ビット\",\n        \"4bit\",\n        \"4-bit\"\n      ],\n      [\n        \"LoRA\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"qlora_paraphrase\",\n    \"category\": \"paraphrase\",\n    \"domain\": \"general\",\n    \"question\": \"QLoRAはどのようにメモリを節約して追加学習しますか？\",\n    \"query\": \"QLoRA\",\n    \"expected_term_id\": \"qlora\",\n    \"reference_answer\": \"ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。\",\n    \"keyword_groups\": [\n      [\n        \"量子化\"\n      ],\n      [\n        \"4ビット\",\n        \"4bit\",\n        \"4-bit\"\n      ],\n      [\n        \"LoRA\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"overfitting_seen\",\n    \"category\": \"seen\",\n    \"domain\": \"general\",\n    \"question\": \"「過学習」とは何ですか？\",\n    \"query\": \"過学習\",\n    \"expected_term_id\": \"overfitting\",\n    \"reference_answer\": \"学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。\",\n    \"keyword_groups\": [\n      [\n        \"学習データ\",\n        \"訓練データ\",\n        \"学習に使った\"\n      ],\n      [\n        \"未知\",\n        \"新しい\",\n        \"未学習\",\n        \"汎化\"\n      ],\n      [\n        \"落ち\",\n        \"低下\",\n        \"悪く\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"overfitting_paraphrase\",\n    \"category\": \"paraphrase\",\n    \"domain\": \"general\",\n    \"question\": \"過学習が起きると、学習に使った問題と未知の問題への対応はどうなりますか？\",\n    \"query\": \"過学習\",\n    \"expected_term_id\": \"overfitting\",\n    \"reference_answer\": \"学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。\",\n    \"keyword_groups\": [\n      [\n        \"学習データ\",\n        \"訓練データ\",\n        \"学習に使った\"\n      ],\n      [\n        \"未知\",\n        \"新しい\",\n        \"未学習\",\n        \"汎化\"\n      ],\n      [\n        \"落ち\",\n        \"低下\",\n        \"悪く\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"lec-start-tag_seen\",\n    \"category\": \"seen\",\n    \"domain\": \"course_specific\",\n    \"question\": \"「lecNN-start タグ」とは何ですか？\",\n    \"query\": \"lecNN-start タグ\",\n    \"expected_term_id\": \"lec-start-tag\",\n    \"reference_answer\": \"AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。\",\n    \"keyword_groups\": [\n      [\n        \"タグ\"\n      ],\n      [\n        \"開始\",\n        \"始め\",\n        \"スタート\"\n      ],\n      [\n        \"演習\",\n        \"コード\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"lec-start-tag_paraphrase\",\n    \"category\": \"paraphrase\",\n    \"domain\": \"course_specific\",\n    \"question\": \"lecNN-start タグを使うと、前の回を終えていなくても何ができますか？\",\n    \"query\": \"lecNN-start タグ\",\n    \"expected_term_id\": \"lec-start-tag\",\n    \"reference_answer\": \"AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。\",\n    \"keyword_groups\": [\n      [\n        \"タグ\"\n      ],\n      [\n        \"開始\",\n        \"始め\",\n        \"スタート\"\n      ],\n      [\n        \"演習\",\n        \"コード\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"homework-levels_seen\",\n    \"category\": \"seen\",\n    \"domain\": \"course_specific\",\n    \"question\": \"「宿題の3レベル」とは何ですか？\",\n    \"query\": \"宿題の3レベル\",\n    \"expected_term_id\": \"homework-levels\",\n    \"reference_answer\": \"AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。\",\n    \"keyword_groups\": [\n      [\n        \"基礎\"\n      ],\n      [\n        \"実装\"\n      ],\n      [\n        \"発展\"\n      ],\n      [\n        \"1つ\",\n        \"一つ\",\n        \"どれか\",\n        \"いずれか\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"homework-levels_paraphrase\",\n    \"category\": \"paraphrase\",\n    \"domain\": \"course_specific\",\n    \"question\": \"宿題の3レベルには何があり、全部取り組む必要がありますか？\",\n    \"query\": \"宿題の3レベル\",\n    \"expected_term_id\": \"homework-levels\",\n    \"reference_answer\": \"AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。\",\n    \"keyword_groups\": [\n      [\n        \"基礎\"\n      ],\n      [\n        \"実装\"\n      ],\n      [\n        \"発展\"\n      ],\n      [\n        \"1つ\",\n        \"一つ\",\n        \"どれか\",\n        \"いずれか\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"answer-md_seen\",\n    \"category\": \"seen\",\n    \"domain\": \"course_specific\",\n    \"question\": \"「answer.md」とは何ですか？\",\n    \"query\": \"answer.md\",\n    \"expected_term_id\": \"answer-md\",\n    \"reference_answer\": \"AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。\",\n    \"keyword_groups\": [\n      [\n        \"演習\",\n        \"宿題\"\n      ],\n      [\n        \"結果\",\n        \"気づき\",\n        \"考察\"\n      ],\n      [\n        \"PR\",\n        \"プルリクエスト\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"answer-md_paraphrase\",\n    \"category\": \"paraphrase\",\n    \"domain\": \"course_specific\",\n    \"question\": \"answer.mdには何を書き、どうやって提出しますか？\",\n    \"query\": \"answer.md\",\n    \"expected_term_id\": \"answer-md\",\n    \"reference_answer\": \"AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。\",\n    \"keyword_groups\": [\n      [\n        \"演習\",\n        \"宿題\"\n      ],\n      [\n        \"結果\",\n        \"気づき\",\n        \"考察\"\n      ],\n      [\n        \"PR\",\n        \"プルリクエスト\"\n      ]\n    ],\n    \"reference_source\": \"data/glossary.json\"\n  },\n  {\n    \"id\": \"dropout_outside\",\n    \"category\": \"not_in_glossary\",\n    \"domain\": \"general\",\n    \"question\": \"「ドロップアウト」とは何ですか？\",\n    \"query\": \"ドロップアウト\",\n    \"expected_term_id\": null,\n    \"reference_answer\": \"学習中にニューロンなどをランダムに無効化し、過学習を抑える正則化手法。\",\n    \"keyword_groups\": [\n      [\n        \"ランダム\",\n        \"確率的\"\n      ],\n      [\n        \"無効\",\n        \"除外\",\n        \"落と\",\n        \"ゼロ\",\n        \"使わない\"\n      ],\n      [\n        \"過学習\",\n        \"正則化\"\n      ]\n    ],\n    \"reference_source\": \"https://www.jmlr.org/papers/v15/srivastava14a.html\"\n  },\n  {\n    \"id\": \"gradient-descent_outside\",\n    \"category\": \"not_in_glossary\",\n    \"domain\": \"general\",\n    \"question\": \"「勾配降下法」とは何ですか？\",\n    \"query\": \"勾配降下法\",\n    \"expected_term_id\": null,\n    \"reference_answer\": \"損失関数の勾配を使い、損失が小さくなる方向へ重みなどのパラメータを繰り返し更新する最適化手法。\",\n    \"keyword_groups\": [\n      [\n        \"勾配\",\n        \"微分\"\n      ],\n      [\n        \"損失\",\n        \"誤差\",\n        \"目的関数\"\n      ],\n      [\n        \"更新\",\n        \"調整\"\n      ],\n      [\n        \"減ら\",\n        \"小さ\",\n        \"最小\",\n        \"下が\",\n        \"減少\"\n      ]\n    ],\n    \"reference_source\": \"https://developers.google.com/machine-learning/crash-course/linear-regression/gradient-descent\"\n  },\n  {\n    \"id\": \"distillation_outside\",\n    \"category\": \"not_in_glossary\",\n    \"domain\": \"general\",\n    \"question\": \"「知識蒸留」とは何ですか？\",\n    \"query\": \"知識蒸留\",\n    \"expected_term_id\": null,\n    \"reference_answer\": \"教師モデルの出力などを使って生徒モデルを学習させ、知識を移す手法。モデルを小さくする用途などに使う。\",\n    \"keyword_groups\": [\n      [\n        \"教師\",\n        \"teacher\"\n      ],\n      [\n        \"生徒\",\n        \"student\",\n        \"小さいモデル\",\n        \"小さなモデル\"\n      ],\n      [\n        \"学習\",\n        \"訓練\"\n      ],\n      [\n        \"出力\",\n        \"予測\",\n        \"分布\",\n        \"ソフトラベル\"\n      ]\n    ],\n    \"reference_source\": \"https://arxiv.org/abs/1503.02531\"\n  }\n]\n",
  "glossary_experiment.py": "\"\"\"第2回・発展編のデータ検証、自然文の評価、用語集検索の比較。\n\nモデルに渡す入力と、採点用の正解を分離する。キーワード判定は一次確認で、\n意味の正しさは回答原文を見て手動判定する。GPU や外部 API は使わない。\n\"\"\"\n\nimport importlib.util\nimport json\nimport time\nimport unicodedata\nfrom pathlib import Path\n\n\nCATEGORIES = (\"seen\", \"paraphrase\", \"not_in_glossary\")\nCATEGORY_LABELS = {\n    \"seen\": \"学習と同じ質問\",\n    \"paraphrase\": \"言い換えた質問\",\n    \"not_in_glossary\": \"用語集にない語\",\n}\n\n\ndef normalize(text: str) -> str:\n    return \"\".join(unicodedata.normalize(\"NFKC\", text).casefold().split())\n\n\ndef build_messages(question: str) -> list[dict]:\n    \"\"\"回答時は質問だけを渡す。用語集・参照回答・判定用語句は入れない。\"\"\"\n    if not isinstance(question, str) or not question.strip():\n        raise ValueError(\"質問は空でない文字列にしてください\")\n    return [{\"role\": \"user\", \"content\": question}]\n\n\ndef load_inputs(folder: Path) -> tuple[dict, list[dict], list[dict]]:\n    glossary = json.loads((folder / \"glossary.json\").read_text(encoding=\"utf-8\"))\n    records = [\n        json.loads(line)\n        for line in (folder / \"glossary_sft.jsonl\").read_text(encoding=\"utf-8\").splitlines()\n        if line.strip()\n    ]\n    questions = json.loads((folder / \"evaluation.json\").read_text(encoding=\"utf-8\"))\n    validate_inputs(glossary, records, questions)\n    return glossary, records, questions\n\n\ndef validate_inputs(glossary: dict, records: list[dict], questions: list[dict]) -> None:\n    terms = glossary[\"terms\"]\n    by_id = {t[\"id\"]: t for t in terms}\n    if len(by_id) != len(terms) or not records or not questions:\n        raise ValueError(\"用語 ID の重複、または空のデータがあります\")\n    train_questions = set()\n    definitions = {t[\"definition\"] for t in terms}\n    for record in records:\n        messages = record[\"messages\"]\n        if [m[\"role\"] for m in messages] != [\"user\", \"assistant\"]:\n            raise ValueError(\"学習データは user と assistant の 2 メッセージです\")\n        if any(not isinstance(m[\"content\"], str) or not m[\"content\"].strip() for m in messages):\n            raise ValueError(\"空の質問・回答があります\")\n        if messages[1][\"content\"] not in definitions:\n            raise ValueError(\"用語集にない定義が学習データに含まれています\")\n        train_questions.add(messages[0][\"content\"])\n    if len(train_questions) != len(records):\n        raise ValueError(\"学習用の質問文が重複しています\")\n\n    all_names = {\n        normalize(name)\n        for term in terms\n        for name in [term[\"term\"], term[\"id\"], *term[\"aliases\"]]\n    }\n    ids, texts = set(), set()\n    for q in questions:\n        if q[\"id\"] in ids or q[\"question\"] in texts:\n            raise ValueError(\"評価問題の ID・質問文が重複しています\")\n        ids.add(q[\"id\"])\n        texts.add(q[\"question\"])\n        if q[\"category\"] not in CATEGORIES:\n            raise ValueError(\"不明な評価カテゴリです\")\n        if not q[\"query\"] or normalize(q[\"query\"]) not in normalize(q[\"question\"]):\n            raise ValueError(\"検索語は問題文に実際に書かれた語にしてください\")\n        groups = q[\"keyword_groups\"]\n        if not groups or any(not g or any(not w for w in g) for g in groups):\n            raise ValueError(\"キーワードの判定条件が空です\")\n        if not isinstance(q[\"reference_answer\"], str) or not q[\"reference_answer\"]:\n            raise ValueError(\"参照回答がありません\")\n        if q[\"category\"] == \"not_in_glossary\":\n            if q[\"expected_term_id\"] is not None or normalize(q[\"query\"]) in all_names:\n                raise ValueError(\"未掲載の語が用語集の名前・別名と一致しています\")\n            if q[\"question\"] in train_questions:\n                raise ValueError(\"未掲載語の質問が学習に混ざっています\")\n        else:\n            term = by_id[q[\"expected_term_id\"]]\n            if q[\"reference_answer\"] != term[\"definition\"]:\n                raise ValueError(\"掲載語の参照回答が用語集と一致しません\")\n            if normalize(q[\"query\"]) not in {\n                normalize(name) for name in [term[\"term\"], term[\"id\"], *term[\"aliases\"]]\n            }:\n                raise ValueError(\"検索語が対象用語の名前・別名に一致しません\")\n        if q[\"category\"] == \"seen\" and q[\"question\"] not in train_questions:\n            raise ValueError(\"seen の質問が学習データにありません\")\n        if q[\"category\"] == \"paraphrase\" and q[\"question\"] in train_questions:\n            raise ValueError(\"言い換え問題が学習データと重複しています\")\n\n\ndef check_keywords(answer: str, groups: list[list[str]]) -> dict:\n    \"\"\"各グループから少なくとも 1 語があるか。意味の正解を保証しない。\"\"\"\n    text = normalize(answer)\n    missing = [group for group in groups if not any(normalize(word) in text for word in group)]\n    return {\"keyword_ok\": not missing, \"missing_keyword_groups\": missing}\n\n\ndef summarize(rows: list[dict], label: str, seconds: float) -> dict:\n    def stats(items):\n        judged = [r for r in items if r.get(\"manual_correct\") is not None]\n        return {\n            \"n\": len(items),\n            \"keyword_rate\": sum(r[\"keyword_ok\"] for r in items) / len(items) if items else None,\n            \"manual_reviewed\": len(judged),\n            # 未判定を誤答や正答として数えない。全問判定後だけ正解率を出す。\n            \"manual_correct_rate\": (\n                sum(r[\"manual_correct\"] for r in judged) / len(items)\n                if items and len(judged) == len(items) else None\n            ),\n        }\n    return {\n        \"label\": label,\n        **stats(rows),\n        \"seconds\": round(seconds, 3),\n        \"by_category\": {category: stats([r for r in rows if r[\"category\"] == category]) for category in CATEGORIES},\n    }\n\n\ndef evaluate_model(generate, questions: list[dict], label: str, output: Path) -> tuple[dict, list[dict]]:\n    rows = []\n    started = time.perf_counter()\n    for q in questions:\n        # generate に渡せるのは質問の文字列だけ。正解は生成後に採点に使う。\n        answer = generate(q[\"question\"])\n        rows.append({\n            \"id\": q[\"id\"], \"category\": q[\"category\"], \"question\": q[\"question\"],\n            \"raw\": answer, \"reference_answer\": q[\"reference_answer\"],\n            **check_keywords(answer, q[\"keyword_groups\"]), \"manual_correct\": None,\n        })\n        print(f\"{label}: {len(rows)}/{len(questions)} {q['id']}\")\n    summary = summarize(rows, label, time.perf_counter() - started)\n    write_json(output, {\"summary\": summary, \"rows\": rows})\n    return summary, rows\n\n\ndef apply_manual_review(rows: list[dict], verdicts: dict) -> list[dict]:\n    known = {r[\"id\"] for r in rows}\n    if set(verdicts) - known:\n        raise ValueError(\"手動判定に不明な問題 ID があります\")\n    if any(value is not None and type(value) is not bool for value in verdicts.values()):\n        raise ValueError(\"手動判定は True / False / None で指定してください\")\n    return [{**r, \"manual_correct\": verdicts.get(r[\"id\"])} for r in rows]\n\n\ndef load_search_function(tools_file: Path, data_dir: Path):\n    \"\"\"既存 app/tools.py をそのまま読み、用語集の保存先だけを差し替える。\"\"\"\n    spec = importlib.util.spec_from_file_location(\"glossary_search_snapshot\", tools_file)\n    if spec is None or spec.loader is None:\n        raise ValueError(\"検索用の Python ファイルを読み込めません\")\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    module.DATA_DIR = data_dir\n    module.load_glossary.cache_clear()\n    return module.search_glossary\n\n\ndef evaluate_search(search, questions: list[dict], glossary: dict, output: Path) -> tuple[dict, list[dict]]:\n    by_name = {t[\"term\"]: t[\"id\"] for t in glossary[\"terms\"]}\n    started, rows = time.perf_counter(), []\n    for q in questions:\n        # query は問題文に書かれた用語。expected_term_id や参照回答は検索に渡さない。\n        raw = search(q[\"query\"])\n        try:\n            hits = json.loads(raw)\n        except json.JSONDecodeError:\n            hits = []\n        first = hits[0] if hits else None\n        found_id = by_name[first[\"term\"]] if first else None\n        rows.append({\n            \"id\": q[\"id\"], \"category\": q[\"category\"], \"question\": q[\"question\"],\n            \"query\": q[\"query\"], \"raw_search_result\": raw,\n            \"raw\": first[\"definition\"] if first else raw,\n            \"hits\": hits, \"found_term_id\": found_id,\n            \"retrieval_ok\": found_id == q[\"expected_term_id\"] if q[\"expected_term_id\"] else None,\n        })\n    listed = [r for r in rows if r[\"category\"] != \"not_in_glossary\"]\n    unlisted = [r for r in rows if r[\"category\"] == \"not_in_glossary\"]\n    summary = {\n        \"label\": \"search\", \"n\": len(rows), \"seconds\": round(time.perf_counter() - started, 3),\n        \"listed_n\": len(listed),\n        \"listed_top1_hit_rate\": sum(r[\"retrieval_ok\"] for r in listed) / len(listed) if listed else None,\n        \"unlisted_n\": len(unlisted),\n        \"unlisted_not_found_rate\": sum(not r[\"hits\"] for r in unlisted) / len(unlisted) if unlisted else None,\n    }\n    write_json(output, {\"summary\": summary, \"rows\": rows})\n    return summary, rows\n\n\ndef write_json(path: Path, value) -> None:\n    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + \"\\n\", encoding=\"utf-8\")\n",
  "search_tools.py": "\"\"\"第3回: エージェントに持たせるツール。\n\nここに書いた関数の docstring と型ヒントが、そのままモデルへの「ツールの説明文」になる。\n説明文を書き換えると、モデルがツールを使うかどうか・どんな引数で呼ぶかが変わる。\n\"\"\"\n\nimport json\nfrom functools import cache\nfrom pathlib import Path\n\nDATA_DIR = Path(__file__).resolve().parent.parent / \"data\"\nMAX_RESULTS = 3\n\n\n@cache\ndef load_glossary() -> list[dict]:\n    with open(DATA_DIR / \"glossary.json\", encoding=\"utf-8\") as f:\n        return json.load(f)[\"terms\"]\n\n\ndef _normalize(text: str) -> str:\n    return text.casefold().replace(\" \", \"\").replace(\"　\", \"\")\n\n\ndef _names(entry: dict) -> list[str]:\n    return [_normalize(entry[\"term\"]), _normalize(entry[\"id\"])] + [\n        _normalize(a) for a in entry[\"aliases\"]\n    ]\n\n\ndef search_glossary(query: str) -> str:\n    \"\"\"講座の用語集から用語を検索し、定義と、どの回で扱うかを返す。\n\n    AIエンジニアリングの用語や、この講座（AIE2026）に固有の言葉について\n    答える前に使うこと。query には調べたい用語を1つだけ入れる（例: \"LoRA\"）。\n    \"\"\"\n    q = _normalize(query)\n    if not q:\n        return \"検索語が空です。調べたい用語を1つ指定してください。\"\n    glossary = load_glossary()\n\n    exact = [e for e in glossary if q in _names(e)]\n    # 「r」のような短い別名が何にでも当たらないよう、名前側が短いものは除く\n    partial = [\n        e for e in glossary\n        if e not in exact and any(q in n or (len(n) >= 3 and n in q) for n in _names(e))\n    ]\n    in_definition = [\n        e for e in glossary\n        if e not in exact and e not in partial and q in _normalize(e[\"definition\"])\n    ]\n    hits = (exact + partial + in_definition)[:MAX_RESULTS]\n\n    if not hits:\n        return f\"用語集に「{query}」は見つかりませんでした。\"\n    return json.dumps(\n        [\n            {\n                \"term\": e[\"term\"],\n                \"lecture\": e[\"lecture\"],\n                \"definition\": e[\"definition\"],\n                \"related\": e[\"related\"],\n            }\n            for e in hits\n        ],\n        ensure_ascii=False,\n    )\n\n\ndef get_lecture_summary(lecture: int) -> str:\n    \"\"\"指定した回（1始まりの整数）の講義の要点を返す。\n\n    「第3回では何を学ぶ？」のように、回の内容や関係を聞かれたときに使う。\n    \"\"\"\n    path = DATA_DIR / \"lectures\" / f\"{lecture:02d}.md\"\n    if not path.exists():\n        available = sorted(p.stem for p in (DATA_DIR / \"lectures\").glob(\"*.md\"))\n        return f\"第{lecture}回の要点はまだありません。用意されている回: {', '.join(available)}\"\n    return path.read_text(encoding=\"utf-8\")\n"
}''')
EXPECTED_SHA256 = {'glossary.json': '4b614a08a754bf6cf714f905f185ad5ef22e0957c7e64434a4b418edb68c2319', 'glossary_sft.jsonl': 'af6ac820e0331d8d9af376e3b4cccff1b31f2f0f8bb462839752e3282bbd1081', 'evaluation.json': '2b068446ed4ea335e0e3fa6d98d0eecd23a762b5c0b6e84c6fd2e2cf41298286', 'glossary_experiment.py': '69b74b4697e17cc52052c3153cc05834af4daf00a3c29282e3d59c7b5a5f6e92', 'search_tools.py': '76fe15419707e59d437d9f834b013a6b9c4c56ae7ad964a25dd94154abf9323e'}

INPUT_DIR = WORK / "inputs"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, text in BUNDLED_FILES.items():
    data = text.encode("utf-8")
    assert hashlib.sha256(data).hexdigest() == EXPECTED_SHA256[name], name
    (INPUT_DIR / name).write_bytes(data)
spec = importlib.util.spec_from_file_location("glossary_experiment", INPUT_DIR / "glossary_experiment.py")
experiment = importlib.util.module_from_spec(spec)
spec.loader.exec_module(experiment)
glossary, train_records, questions = experiment.load_inputs(INPUT_DIR)
search_glossary = experiment.load_search_function(INPUT_DIR / "search_tools.py", INPUT_DIR)
print(f"用語数={len(glossary['terms'])} / 学習例={len(train_records)} / 評価問題={len(questions)}")
print("学習例:", json.dumps(train_records[0], ensure_ascii=False, indent=2))

用語数=58 / 学習例=116 / 評価問題=15
学習例: {
  "messages": [
    {
      "role": "user",
      "content": "「AIエンジニアリング」とは何ですか？"
    },
    {
      "role": "assistant",
      "content": "学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。"
    }
  ]
}


In [38]:
import pandas as pd
pd.set_option("display.max_colwidth", None)
display(pd.DataFrame([{
    "id": q["id"], "種類": experiment.CATEGORY_LABELS[q["category"]],
    "質問": q["question"], "検索語": q["query"], "参照回答": q["reference_answer"],
} for q in questions]))
print("検索例:", search_glossary("LoRA"))

,id,種類,質問,検索語,参照回答
0,lora_seen,学習と同じ質問,「LoRA」とは何ですか？,LoRA,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。
1,lora_paraphrase,言い換えた質問,LoRAでは元の重みをどう扱い、何を学習しますか？,LoRA,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。
2,qlora_seen,学習と同じ質問,「QLoRA」とは何ですか？,QLoRA,ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。
3,qlora_paraphrase,言い換えた質問,QLoRAはどのようにメモリを節約して追加学習しますか？,QLoRA,ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。
4,overfitting_seen,学習と同じ質問,「過学習」とは何ですか？,過学習,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。
5,overfitting_paraphrase,言い換えた質問,過学習が起きると、学習に使った問題と未知の問題への対応はどうなりますか？,過学習,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。
6,lec-start-tag_seen,学習と同じ質問,「lecNN-start タグ」とは何ですか？,lecNN-start タグ,AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。
7,lec-start-tag_paraphrase,言い換えた質問,lecNN-start タグを使うと、前の回を終えていなくても何ができますか？,lecNN-start タグ,AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。
8,homework-levels_seen,学習と同じ質問,「宿題の3レベル」とは何ですか？,宿題の3レベル,AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。
9,homework-levels_paraphrase,言い換えた質問,宿題の3レベルには何があり、全部取り組む必要がありますか？,宿題の3レベル,AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。


検索例: [{"term": "LoRA", "lecture": 2, "definition": "事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。", "related": ["peft", "lora-rank", "qlora"]}, {"term": "ランク（LoRAのr）", "lecture": 2, "definition": "LoRAで使う低ランク行列の大きさを決めるハイパーパラメータ。大きいほど表現力が上がるが、学習するパラメータとメモリも増える。", "related": ["lora"]}, {"term": "QLoRA", "lecture": 2, "definition": "ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。", "related": ["lora", "quantization"]}]


## 3. モデルの読み込みと学習前の回答

モデルに渡すのは `[{"role": "user", "content": 質問}]` だけです。
正解は生成後に確認します。キーワードの一致率は一次確認であり、内容正解率ではありません。
否定や矛盾があっても語句だけは含められるので、セクション 6 で原文も読みます。

In [39]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
if not torch.cuda.is_available():
    raise RuntimeError("このノートブックは Colab の CUDA GPU を使って実行してください。")
set_seed(CONFIG["seed"])
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
lengths = [len(tok.apply_chat_template(r["messages"], tokenize=True, enable_thinking=False))
           for r in train_records]
print("学習例の最大トークン数:", max(lengths), "/ 上限:", CONFIG["max_seq_len"])
if max(lengths) > CONFIG["max_seq_len"]:
    raise ValueError("定義文を切り捨てないよう max_seq_len を増やして、上から実行し直してください。")
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32,
).to("cuda")
model.config.use_cache = True
model.eval()
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| 重みのメモリ(GB):", BASE_MEMORY_GB)

def generate_answer(model, question):
    messages = experiment.build_messages(question)
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"],
                             pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

before_summary, before_rows = experiment.evaluate_model(
    lambda q: generate_answer(model, q), questions, "before", WORK / "eval_before.json",
)
print(json.dumps(before_summary, ensure_ascii=False, indent=2))
display(pd.DataFrame(before_rows)[["id", "question", "raw", "keyword_ok"]])

学習例の最大トークン数: 2 / 上限: 512


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | 重みのメモリ(GB): 3.01
before: 1/15 lora_seen
before: 2/15 lora_paraphrase
before: 3/15 qlora_seen
before: 4/15 qlora_paraphrase
before: 5/15 overfitting_seen
before: 6/15 overfitting_paraphrase
before: 7/15 lec-start-tag_seen
before: 8/15 lec-start-tag_paraphrase
before: 9/15 homework-levels_seen
before: 10/15 homework-levels_paraphrase
before: 11/15 answer-md_seen
before: 12/15 answer-md_paraphrase
before: 13/15 dropout_outside
before: 14/15 gradient-descent_outside
before: 15/15 distillation_outside
{
  "label": "before",
  "n": 15,
  "keyword_rate": 0.0,
  "manual_reviewed": 0,
  "manual_correct_rate": null,
  "seconds": 178.164,
  "by_category": {
    "seen": {
      "n": 6,
      "keyword_rate": 0.0,
      "manual_reviewed": 0,
      "manual_correct_rate": null
    },
    "paraphrase": {
      "n": 6,
      "keyword_rate": 0.0,
      "manual_reviewed": 0,
      "manual_correct_rate": null
    },
    "not_in_glossary": {
      "n": 3,
      "keyword_rate": 0.

,id,question,raw,keyword_ok
0,lora_seen,「LoRA」とは何ですか？,**LoRA** は、大規模言語モデル（LLM）のトレーニングにおいて非常に重要な技術の一つです。\n\n簡単に言うと、**「ローカルなモデル（小さなモデル）を、大規模なモデル（大きなモデル）に追加して、より高い精度や表現力を得るための手法」**です。\n\n具体的には、以下の 3 つのステップで考えられます：\n\n1. **ローカルモデルの作成**：\n 研究者が、特定の分野（例：数学、言語、科学）に特化した、非常に小さなモデルを作成します。このモデルは、大規模モデルに比べて計算コストが非常に低く、学習速度も速いです。\n\n2. **LoRA の導入**：\n 大規模モデル（例えば GPT-3.5 や Llama-3）にこのローカルモデルを「追加」して学習させます。この際、大規模モデルはローカルモデルの知識を学習し、その知識をそのまま大規模モデルに適用します。\n\n3. **LoRA の調整**：\n 学習过程中、LoRA によって学習された知識を、大規模モデルに「追加」して、より高い精度や表現力を得ることができます。,False
1,lora_paraphrase,LoRAでは元の重みをどう扱い、何を学習しますか？,LoRA（Low-Rank Adaptation）は、元のモデル（通常は大規模な Transformer などの基盤モデル）の重みを**「ゼロ」または「非常に小さい」**に設定し、学習する際に**「元の重み」を維持しつつ、新しい知識や特徴を学習する**という仕組みです。\n\n具体的には以下の 2 つの側面が考えられます。\n\n### 1. 元の重みをどのように扱うか？\nLoRA では、元のモデルの重み（パラメータ）は**学習対象（学習目標）として扱いません**。\n\n* **学習対象は「新しい特徴」**です。LoRA 学習では、元のモデルの重みは**0**（または非常に小さい値）に設定されます。\n* **学習目標は「元の重み」を維持する**ことです。つまり、学習する際に元のモデルの重みは**0**に設定され、学習する際に元の重みは**0**に設定されます。\n* **結果として**：学習する際に元の重みは**0**に設定され、学習する際に元の重みは**0**に設定されます。\n\n### 2. 何を学習しますか？,False
2,qlora_seen,「QLoRA」とは何ですか？,**QLoRA** は、**Qwen3.5** という大規模言語モデル（LLM）の一種です。\n\nQwen3.5 は、Qwen3.5 の進化版であり、**Qwen3.5 の 1.5% のモデルサイズ**です。このモデルは、Qwen3.5 の特徴を維持しつつ、より**軽量**かつ**高速**なパフォーマンスを実現しています。\n\n主な特徴としては以下の通りです：\n\n* **軽量なサイズ**: Qwen3.5 の 1.5% であるため、メモリ使用量や計算コストが大幅に削減され、特に**モバイル端末**や**軽量なサーバー**で動作しやすくなります。\n* **高速な処理**: 計算速度が Qwen3.5 よりも速く、特に**短文字の質問**や**複雑な計算**に対して処理がスムーズに行えます。\n* **多言語対応**: 日本語以外の言語（英語、中国語、韓国語、フランス語、ドイツ語、スペイン語、イタリア語、ロシア語、スロベニア語、ポーランド語、ポルトガル語、オランダ語、スウェーデン語、ドイツ語,False
3,qlora_paraphrase,QLoRAはどのようにメモリを節約して追加学習しますか？,QLoRA（Quantized Large Language Model）は、従来の大規模言語モデル（LLM）に比べて**メモリ効率を大幅に向上**させることで、追加学習（Fine-tuning）やトレーニングの速度を劇的に加速します。\n\nその仕組みは主に以下の 3 つのステップで達成されています。\n\n### 1. モデルの圧縮（Quantization）\nQLoRA は、モデルの参数を **16 進数制（16-bit）** に圧縮します。\n* **従来のモデル**: 1 個の参数は 4 字节（4 bytes）とみなされ、メモリは約 4GB 程度になります。\n* **QLoRA モデル**: 1 個の参数は **1 字节（1 byte）** に変更され、メモリは約 1GB 程度になります。\n* **効果**: 1 個の参数を 4 倍に減らすことで、同じ出力を生成する際に必要なメモリの半分程度になります。\n\n### 2. 計算精度の維持（Precision）\n圧縮されたモデル（16-bit）を学習する際、精度が失われると学習が失敗します。そのため,False
4,overfitting_seen,「過学習」とは何ですか？,**過学習**（Overfitting）とは、機械学習や統計学習において、モデルが学習したデータに**過剰に適合**し、**真のデータに欠ける**現象を指します。\n\n簡単に言うと、「学習したことが全部で、真の答えも全部で」という状態です。\n\n具体的には以下の 3 つの要素が重なって起こります。\n\n1. **過学習**：モデルが学習したデータに**過剰に適合**している。\n2. **欠学習**：モデルが学習したデータに**欠けている**（真のデータに欠ける）。\n3. **過学習・欠学習**：両者の両方がある。\n\nこの状態は、モデルが「学習したデータ」に**過剰に適合**しているため、**真のデータに欠ける**状態を指します。\n\n### 過学習の例\n例えば、「猫は犬ではない」という事実を学習したモデル（「猫は犬ではない」というルールをすべて適用した）が、真の猫と犬のデータに欠けるという状態です。\n\n### 解決策\n過学習を防ぐために、以下の方法が一般的です。\n*,False
5,overfitting_paraphrase,過学習が起きると、学習に使った問題と未知の問題への対応はどうなりますか？,過学習（Overfitting）が起きると、学習に使った問題と未知の問題への対応は**「学習したパターンをそのまま適用する」のではなく、学習したデータに特化した「特化されたモデル」を構築し、その特化されたモデルが未知の問題に対して**\n\n* **学習した問題の対応**: 依然として、学習したデータに特化したパターンを適用します。\n* **未知の問題への対応**: 学習したデータに特化したモデルが、学習した問題と未知の問題の間に埋もれた「未知の問題のデータ」を学習し、その特化されたモデルが未知の問題に対して**より良い対応**（精度が向上する）をします。\n\nつまり、過学習は学習した問題に対する精度を高めるために生じますが、その結果、学習した問題と未知の問題の間に埋もれた「未知の問題のデータ」を学習し、その特化されたモデルが未知の問題に対して**より良い対応**をします。\n\nこの現象は、**「学習した問題と未知の問題の間に埋もれたデータ」を学習し、その特化されたモデルが未知の問題に対してより良い対応をします**。,False
6,lec-start-tag_seen,「lecNN-start タグ」とは何ですか？,「lecNN-start」は、**LeetCode（リテール・コード・ネスト）の「LeetCode」のスタートアッププロジェクト**です。\n\nこのプロジェクトは、LeetCode の開発者コミュニティ（GitHub）を運営する「LeetCode 株式会社」が、2024 年 1 月 1 日に設立したものです。主な目的は、LeetCode の開発者コミュニティをより広く、かつ質の高いコミュニティに引き込むことで、LeetCode の発展と活発な開発活動を促進することです。\n\nこのプロジェクトは、以下の目的で活動しています：\n* LeetCode の開発者コミュニティの拡大と活性化\n* LeetCode の開発者コミュニティの質を高める\n* LeetCode の開発者コミュニティの発展を促進する\n\nLeetCode 株式会社は、この「lecNN-start」プロジェクトを通じて、LeetCode の開発者コミュニティをより活発に発展させることを目指しています。,False
7,lec-start-tag_paraphrase,lecNN-start タグを使うと、前の回を終えていなくても何ができますか？,**lecNN-start タグを使うと、前の回を終えていなくても、学習を再開できます。**\n\nこの機能は、学習の進捗状況（学習済み項目数、学習済み時間など）を記録し、次の回に自動的に「学習済み項目数」や「学習済み時間」を基準に、学習を再開する仕組みです。\n\n具体的には以下のようになります。\n1. **学習済み項目数**が 0 になるまで、学習を続けることができます。\n2. **学習済み時間**が 0 になるまで、学習を続けることができます。\n3. 学習を再開する際、学習済み項目数や学習済み時間が 0 になるまで、学習を続けることができます。\n\nつまり、前の回を終わらせたとしても、学習の進行状況は残っており、次の回に自動的に学

## 4. 用語集 Q&A で LoRA 学習

116 件の `messages` をそのまま `Dataset.from_list` に渡します。
図書館版の FAQ や JSON のお手本は使いません。未掲載語の参照回答も学習には使いません。
学習時間・最終ログの loss・ステップ数を変数に残し、セクション 7 で保存します。
学習できなかった場合は、学習後の結果を作らず、実行できた範囲を記録してください。

In [40]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

train_ds = Dataset.from_list(train_records)
model.config.use_cache = False
lora = LoraConfig(
    r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
TRAIN_LOGGING_STEPS = 5
sft = SFTConfig(
    output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=4, gradient_accumulation_steps=2,
    learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"],
    logging_steps=TRAIN_LOGGING_STEPS, save_steps=10, save_total_limit=2,
    fp16=False, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"],
)
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds,
                     peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
started = time.perf_counter()
trainer.train()
TRAINING_SECONDS = round(time.perf_counter() - started, 1)
LOSS_HISTORY = [{"step": row["step"], "training_loss": row["loss"]}
                for row in trainer.state.log_history if "loss" in row]
LAST_LOGGED_TRAINING_LOSS = LOSS_HISTORY[-1]["training_loss"] if LOSS_HISTORY else None
print(f"学習時間: {TRAINING_SECONDS} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)
tuned = trainer.model
tuned.config.use_cache = True
tuned.eval();

Tokenizing train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
5,3.705675
10,2.849201
15,2.501185
20,1.926214
25,1.753987
30,1.628736
35,1.079296
40,1.013886
45,0.808279
50,0.544853


学習時間: 179.0 秒 | ステップ数: 75 | エポック: 5.0


## 5. 学習後と用語集検索を比較する

学習後も質問だけを渡します。検索版は、問題文に書かれた用語を `query` として既存の検索関数に渡し、
先頭に返った定義を表示します。自然文から検索語を抽出する能力や、意味検索の性能は測りません。
検索に参照回答や正解 ID は渡しません。候補は最大 3 件で、全候補も結果ファイルに残します。

モデルの「キーワード一致率」と、検索の「正しい用語を先頭に見つけた割合」は別の指標です。
検索にない語は「見つからない」が期待されますが、モデルが正しく説明できる可能性はあります。

In [41]:
after_summary, after_rows = experiment.evaluate_model(
    lambda q: generate_answer(tuned, q), questions, "after", WORK / "eval_after.json",
)
search_summary, search_rows = experiment.evaluate_search(
    search_glossary, questions, glossary, WORK / "eval_search.json",
)
display(pd.DataFrame([{
    "条件": summary["label"],
    "同じ質問のキーワード一致率": summary["by_category"]["seen"]["keyword_rate"],
    "言い換えのキーワード一致率": summary["by_category"]["paraphrase"]["keyword_rate"],
    "未掲載語のキーワード一致率": summary["by_category"]["not_in_glossary"]["keyword_rate"],
    "回答生成時間（秒）": summary["seconds"],
} for summary in [before_summary, after_summary]]))
print("検索結果:", json.dumps(search_summary, ensure_ascii=False, indent=2))
comparison_rows = [{
    "id": q["id"], "種類": experiment.CATEGORY_LABELS[q["category"]], "質問": q["question"],
    "学習前": b["raw"], "学習後": a["raw"], "検索": s["raw"], "参照回答": q["reference_answer"],
} for q, b, a, s in zip(questions, before_rows, after_rows, search_rows)]
display(pd.DataFrame(comparison_rows))

after: 1/15 lora_seen
after: 2/15 lora_paraphrase
after: 3/15 qlora_seen
after: 4/15 qlora_paraphrase
after: 5/15 overfitting_seen
after: 6/15 overfitting_paraphrase
after: 7/15 lec-start-tag_seen
after: 8/15 lec-start-tag_paraphrase
after: 9/15 homework-levels_seen
after: 10/15 homework-levels_paraphrase
after: 11/15 answer-md_seen
after: 12/15 answer-md_paraphrase
after: 13/15 dropout_outside
after: 14/15 gradient-descent_outside
after: 15/15 distillation_outside


,条件,同じ質問のキーワード一致率,言い換えのキーワード一致率,未掲載語のキーワード一致率,回答生成時間（秒）
0,before,0.000000,0.0,0.0,178.164
1,after,0.333333,0.0,0.0,59.851


検索結果: {
  "label": "search",
  "n": 15,
  "seconds": 0.006,
  "listed_n": 12,
  "listed_top1_hit_rate": 1.0,
  "unlisted_n": 3,
  "unlisted_not_found_rate": 1.0
}


,id,種類,質問,学習前,学習後,検索,参照回答
0,lora_seen,学習と同じ質問,「LoRA」とは何ですか？,**LoRA** は、大規模言語モデル（LLM）のトレーニングにおいて非常に重要な技術の一つです。\n\n簡単に言うと、**「ローカルなモデル（小さなモデル）を、大規模なモデル（大きなモデル）に追加して、より高い精度や表現力を得るための手法」**です。\n\n具体的には、以下の 3 つのステップで考えられます：\n\n1. **ローカルモデルの作成**：\n 研究者が、特定の分野（例：数学、言語、科学）に特化した、非常に小さなモデルを作成します。このモデルは、大規模モデルに比べて計算コストが非常に低く、学習速度も速いです。\n\n2. **LoRA の導入**：\n 大規模モデル（例えば GPT-3.5 や Llama-3）にこのローカルモデルを「追加」して学習させます。この際、大規模モデルはローカルモデルの知識を学習し、その知識をそのまま大規模モデルに適用します。\n\n3. **LoRA の調整**：\n 学習过程中、LoRA によって学習された知識を、大規模モデルに「追加」して、より高い精度や表現力を得ることができます。,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習する手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。
1,lora_paraphrase,言い換えた質問,LoRAでは元の重みをどう扱い、何を学習しますか？,LoRA（Low-Rank Adaptation）は、元のモデル（通常は大規模な Transformer などの基盤モデル）の重みを**「ゼロ」または「非常に小さい」**に設定し、学習する際に**「元の重み」を維持しつつ、新しい知識や特徴を学習する**という仕組みです。\n\n具体的には以下の 2 つの側面が考えられます。\n\n### 1. 元の重みをどのように扱うか？\nLoRA では、元のモデルの重み（パラメータ）は**学習対象（学習目標）として扱いません**。\n\n* **学習対象は「新しい特徴」**です。LoRA 学習では、元のモデルの重みは**0**（または非常に小さい値）に設定されます。\n* **学習目標は「元の重み」を維持する**ことです。つまり、学習する際に元のモデルの重みは**0**に設定され、学習する際に元の重みは**0**に設定されます。\n* **結果として**：学習する際に元の重みは**0**に設定され、学習する際に元の重みは**0**に設定されます。\n\n### 2. 何を学習しますか？,2つの行列の積（重み×入力）で誤差を生成し、その誤差にだけ小さな学習率で重みを更新する、という「学習する領域と更新する量」を絞る手法。本講座の第2回演習ではLoRAを扱う。,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。
2,qlora_seen,学習と同じ質問,「QLoRA」とは何ですか？,**QLoRA** は、**Qwen3.5** という大規模言語モデル（LLM）の一種です。\n\nQwen3.5 は、Qwen3.5 の進化版であり、**Qwen3.5 の 1.5% のモデルサイズ**です。このモデルは、Qwen3.5 の特徴を維持しつつ、より**軽量**かつ**高速**なパフォーマンスを実現しています。\n\n主な特徴としては以下の通りです：\n\n* **軽量なサイズ**: Qwen3.5 の 1.5% であるため、メモリ使用量や計算コストが大幅に削減され、特に**モバイル端末**や**軽量なサーバー**で動作しやすくなります。\n* **高速な処理**: 計算速度が Qwen3.5 よりも速く、特に**短文字の質問**や**複雑な計算**に対して処理がスムーズに行えます。\n* **多言語対応**: 日本語以外の言語（英語、中国語、韓国語、フランス語、ドイツ語、スペイン語、イタリア語、ロシア語、スロベニア語、ポーランド語、ポルトガル語、オランダ語、スウェーデン語、ドイツ語,ベースモデルの重みを4ビットや1ビットで圧縮したメモリ使用量を減った後で、LoRAで学習する手法。より大きなモデルを小さなメモリで扱える。,ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。,ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。
3,qlora_paraphrase,言い換えた質問,QLoRAはどのようにメモリを節約して追加学習しますか？,QLoRA（Quantized Large Language Model）は、従来の大規模言語モデル（LLM）に比べて**メモリ効率を大幅に向上**させることで、追加学習（Fine-tuning）やトレーニングの速度を劇的に加速します。\n\nその仕組みは主に以下の 3 つのステップで達成されています。\n\n### 1. モデルの圧縮（Quantization）\nQLoRA は、モデルの参数を **16 進数制（16-bit）** に圧縮します。\n* **従来のモデル**: 1 個の参数は 4 字节（4 bytes）とみなされ、メモリは約 4GB 程度になります。\n* **QLoRA モデル**: 1 個の参数は **1 字节（1 byte）** に変更され、メモリは約 1GB 程度になります。\n* **効果**: 1 個の参数を 4 倍に減らすことで、同じ出力を生成する際に必要なメモリの半分程度になります。\n\n### 2. 計算精度の維持（Precision）\n圧縮されたモデル（16-bit）を学習する際、精度が失われると学習が失敗します。そのため,1 つの重み（2ビットで表す）でモデルを拡張できる。同じ入力・出力で、1ビットで表す重みを2ビットにするとメモリが2倍、2ビットで表す重みを1ビットにするとメモリが半分になる。同じ入力・出力で、1ビットで表す重みを2ビットにするとメモリが2倍、2ビットで表す重みを1ビットにするとメモリが半分になる。同じ入力・出力で、1ビットで表す重みを2ビットにするとメモリが2倍、2ビットで表す重みを1ビットにするとメモリが半分になる。,ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。,ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。
4,overfitting_seen,学習と同じ質問,「過学習」とは何ですか？,**過学習**（Overfitting）とは、機械学習や統計学習において、モデルが学習したデータに**過剰に適合**し、**真のデータに欠ける**現象を指します。\n\n簡単に言うと、「学習したことが全部で、真の答えも全部で」という状態です。\n\n具体的には以下の 3 つの要素が重なって起こります。\n\n1. **過学習**：モデルが学習したデータに**過剰に適合**している。\n2. **欠学習**：モデルが学習したデータに**欠けている**（真のデータに欠ける）。\n3. **過学習・欠学習**：両者の両方がある。\n\nこの状態は、モデルが「学習したデータ」に**過剰に適合**しているため、**真のデータに欠ける**状態を指します。\n\n### 過学習の例\n例えば、「猫は犬ではない」という事

## 6. 原文を読んで内容を確認する

参照回答はお手本です。一字一句の一致ではなく、意味が合っているか、誤った説明が混ざっていないかを確認します。
「学習と同じ質問」での改善と「言い換え」での改善を分けて見てください。
未掲載語では、正しい説明・分からないという応答・誤った断定を区別します。
元のモデルが知っていた場合もあるため、未掲載という理由だけで誤答にしません。

下の `MANUAL_VERDICTS` は任意で記入できます。例: `"lora_seen": True`。
`True` は内容が正しい、`False` は誤りまたは回答不足です。未確認は書かないか `None` にします。
正しい保留と誤った断定の違いは原文にも記録されています。
**全問の手動確認が終わるまでは、内容正解率を `null`（未確定）として保存します。**
手動確認を後で行う場合は、このセルとセクション 7 だけ再実行できます。

In [42]:
MANUAL_VERDICTS = {
    "before": {},  # 例: "lora_seen": True
    "after": {},
}
before_rows = experiment.apply_manual_review(before_rows, MANUAL_VERDICTS["before"])
after_rows = experiment.apply_manual_review(after_rows, MANUAL_VERDICTS["after"])
before_summary = experiment.summarize(before_rows, "before", before_summary["seconds"])
after_summary = experiment.summarize(after_rows, "after", after_summary["seconds"])
display(pd.DataFrame([{
    "id": q["id"], "質問": q["question"], "学習前": b["raw"], "学習後": a["raw"],
    "参照回答": q["reference_answer"], "学習前の手動判定": b["manual_correct"],
    "学習後の手動判定": a["manual_correct"],
} for q, b, a in zip(questions, before_rows, after_rows)]))
keyword_regressed = [b["id"] for b, a in zip(before_rows, after_rows)
                     if b["keyword_ok"] and not a["keyword_ok"]]
print("キーワード判定が通過から不通過になった問題（意味の退行かは原文確認）:", keyword_regressed)

,id,質問,学習前,学習後,参照回答,学習前の手動判定,学習後の手動判定
0,lora_seen,「LoRA」とは何ですか？,**LoRA** は、大規模言語モデル（LLM）のトレーニングにおいて非常に重要な技術の一つです。\n\n簡単に言うと、**「ローカルなモデル（小さなモデル）を、大規模なモデル（大きなモデル）に追加して、より高い精度や表現力を得るための手法」**です。\n\n具体的には、以下の 3 つのステップで考えられます：\n\n1. **ローカルモデルの作成**：\n 研究者が、特定の分野（例：数学、言語、科学）に特化した、非常に小さなモデルを作成します。このモデルは、大規模モデルに比べて計算コストが非常に低く、学習速度も速いです。\n\n2. **LoRA の導入**：\n 大規模モデル（例えば GPT-3.5 や Llama-3）にこのローカルモデルを「追加」して学習させます。この際、大規模モデルはローカルモデルの知識を学習し、その知識をそのまま大規模モデルに適用します。\n\n3. **LoRA の調整**：\n 学習过程中、LoRA によって学習された知識を、大規模モデルに「追加」して、より高い精度や表現力を得ることができます。,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習する手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,None,None
1,lora_paraphrase,LoRAでは元の重みをどう扱い、何を学習しますか？,LoRA（Low-Rank Adaptation）は、元のモデル（通常は大規模な Transformer などの基盤モデル）の重みを**「ゼロ」または「非常に小さい」**に設定し、学習する際に**「元の重み」を維持しつつ、新しい知識や特徴を学習する**という仕組みです。\n\n具体的には以下の 2 つの側面が考えられます。\n\n### 1. 元の重みをどのように扱うか？\nLoRA では、元のモデルの重み（パラメータ）は**学習対象（学習目標）として扱いません**。\n\n* **学習対象は「新しい特徴」**です。LoRA 学習では、元のモデルの重みは**0**（または非常に小さい値）に設定されます。\n* **学習目標は「元の重み」を維持する**ことです。つまり、学習する際に元のモデルの重みは**0**に設定され、学習する際に元の重みは**0**に設定されます。\n* **結果として**：学習する際に元の重みは**0**に設定され、学習する際に元の重みは**0**に設定されます。\n\n### 2. 何を学習しますか？,2つの行列の積（重み×入力）で誤差を生成し、その誤差にだけ小さな学習率で重みを更新する、という「学習する領域と更新する量」を絞る手法。本講座の第2回演習ではLoRAを扱う。,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,None,None
2,qlora_seen,「QLoRA」とは何ですか？,**QLoRA** は、**Qwen3.5** という大規模言語モデル（LLM）の一種です。\n\nQwen3.5 は、Qwen3.5 の進化版であり、**Qwen3.5 の 1.5% のモデルサイズ**です。このモデルは、Qwen3.5 の特徴を維持しつつ、より**軽量**かつ**高速**なパフォーマンスを実現しています。\n\n主な特徴としては以下の通りです：\n\n* **軽量なサイズ**: Qwen3.5 の 1.5% であるため、メモリ使用量や計算コストが大幅に削減され、特に**モバイル端末**や**軽量なサーバー**で動作しやすくなります。\n* **高速な処理**: 計算速度が Qwen3.5 よりも速く、特に**短文字の質問**や**複雑な計算**に対して処理がスムーズに行えます。\n* **多言語対応**: 日本語以外の言語（英語、中国語、韓国語、フランス語、ドイツ語、スペイン語、イタリア語、ロシア語、スロベニア語、ポーランド語、ポルトガル語、オランダ語、スウェーデン語、ドイツ語,ベースモデルの重みを4ビットや1ビットで圧縮したメモリ使用量を減った後で、LoRAで学習する手法。より大きなモデルを小さなメモリで扱える。,ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。,None,None
3,qlora_paraphrase,QLoRAはどのようにメモリを節約して追加学習しますか？,QLoRA（Quantized Large Language Model）は、従来の大規模言語モデル（LLM）に比べて**メモリ効率を大幅に向上**させることで、追加学習（Fine-tuning）やトレーニングの速度を劇的に加速します。\n\nその仕組みは主に以下の 3 つのステップで達成されています。\n\n### 1. モデルの圧縮（Quantization）\nQLoRA は、モデルの参数を **16 進数制（16-bit）** に圧縮します。\n* **従来のモデル**: 1 個の参数は 4 字节（4 bytes）とみなされ、メモリは約 4GB 程度になります。\n* **QLoRA モデル**: 1 個の参数は **1 字节（1 byte）** に変更され、メモリは約 1GB 程度になります。\n* **効果**: 1 個の参数を 4 倍に減らすことで、同じ出力を生成する際に必要なメモリの半分程度になります。\n\n### 2. 計算精度の維持（Precision）\n圧縮されたモデル（16-bit）を学習する際、精度が失われると学習が失敗します。そのため,1 つの重み（2ビットで表す）でモデルを拡張できる。同じ入力・出力で、1ビットで表す重みを2ビットにするとメモリが2倍、2ビットで表す重みを1ビットにするとメモリが半分になる。同じ入力・出力で、1ビットで表す重みを2ビットにするとメモリが2倍、2ビットで表す重みを1ビットにするとメモリが半分になる。同じ入力・出力で、1ビットで表す重みを2ビットにするとメモリが2倍、2ビットで表す重みを1ビットにするとメモリが半分になる。,ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。,None,None
4,overfitting_seen,「過学習」とは何ですか？,**過学習**（Overfitting）とは、機械学習や統計学習において、モデルが学習したデータに**過剰に適合**し、**真のデータに欠ける**現象を指します。\n\n簡単に言うと、「学習したことが全部で、真の答えも全部で」という状態です。\n\n具体的には以下の 3 つの要素が重なって起こります。\n\n1. **過学習**：モデルが学習したデータに**過剰に適合**している。\n2. **欠学習**：モデルが学習したデータに**欠けている**（真のデータに欠ける）。\n3. **過学習・欠学習**：両者の両方がある。\n\nこの状態は、モデルが「学習したデータ」に**過剰に適合**しているため、**真のデータに欠ける**状態を指します。\n\n### 過学習の例\n例えば、「猫は犬ではない」という事実を学習したモデル（「猫は犬ではない」というルールをすべて適用した）が、真の猫と犬のデータに欠けるという状態です。\n\n### 解決策\n過学習を防ぐために、以下の方法が一般的です。\n*,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。,None,None
5,overfitting_paraphrase,過学習が起きると、学習に使った問題と未知の問題への対応はどうなりますか？,過学習（Overfitting）が起きると、学習に使った問題と未知の問題への対応は**「学習したパターンをそのまま適用する」のではなく、学習したデータに特化した「特化されたモデル」を構築し、その特化されたモデルが未知の問題に対して**\n\n* **学習した問題の対応**: 依然として、学習したデータに特化したパターンを適用します。\n* **未知の問題への対応**: 学習したデータに特化したモデルが、学習した問題と未知の問題の

キーワード判定が通過から不通過になった問題（意味の退行かは原文確認）: []


## 7. 結果を保存する

`result.json` に条件・学習時間・loss・学習前後の要約・検索の要約を記録します。
各回答原文は `eval_before.json` / `eval_after.json` / `eval_search.json`、
評価問題と正解は `evaluation.json`、手動判定は `manual_review.json` に保存します。

保存先は `/content/glossary_lora_lab` です。実行後はノートブックを保存し、結果ファイルもローカルに保存してください。
提出用には `lectures/02/advanced/` に置きます。`USE_DRIVE=True` の場合は Drive にもコピーします。

考察では、定義の正確さ・言い換えへの対応・未掲載語の応答を比較し、用語集を更新したときに
「再学習する方法」と「検索するデータを書き換える方法」で何が違うかを説明します。
この小さな評価だけで、全用語・全質問への正しさを保証することはできません。

In [43]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in {"work_dir", "drive_dir"}},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）", "dataset_topic": "講座の用語集",
    "answer_style": "自然文", "model_receives_reference_at_inference": False,
    "dataset": {"terms": len(glossary["terms"]), "train_examples": len(train_records),
                "evaluation_questions": len(questions), "input_sha256": EXPECTED_SHA256},
    "evaluation_notes": {
        "keyword_rate": "語句による一次確認。意味の内容正解率ではない。",
        "manual_correct_rate": "全問を手動確認するまで null。",
        "seen": "学習と同じ質問。未知の質問への汎化とは区別する。",
        "not_in_glossary": "未掲載でも正しく説明できれば正解。検索の未検出とは別に評価する。",
        "search_query": "問題文に書かれた用語。正解 ID や参照回答は検索に渡さない。",
    },
    "training": {"seconds": TRAINING_SECONDS, "global_step": trainer.state.global_step,
                 "epochs": trainer.state.epoch, "last_logged_training_loss": LAST_LOGGED_TRAINING_LOSS,
                 "loss_history": LOSS_HISTORY, "logging_steps": TRAIN_LOGGING_STEPS},
    "before": before_summary, "after": after_summary, "search": search_summary,
    "keyword_regressed": keyword_regressed, "executed_training_myself": True,
}
experiment.write_json(WORK / "result.json", result)
experiment.write_json(WORK / "eval_before.json", {"summary": before_summary, "rows": before_rows})
experiment.write_json(WORK / "eval_after.json", {"summary": after_summary, "rows": after_rows})
experiment.write_json(WORK / "evaluation.json", questions)
experiment.write_json(WORK / "manual_review.json", MANUAL_VERDICTS)
experiment.write_json(WORK / "comparison.json", comparison_rows)
if USE_DRIVE:
    import shutil
    target = Path(CONFIG["drive_dir"])
    for path in WORK.glob("*.json"):
        shutil.copy2(path, target / path.name)
    shutil.copytree(INPUT_DIR, target / "inputs", dirs_exist_ok=True)
    shutil.copytree(ADAPTER_DIR, target / "adapter", dirs_exist_ok=True)
print(json.dumps(result, ensure_ascii=False, indent=2))
print("保存先:", WORK)

{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 5,
    "warmup_steps": 6,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,
    "gen": {
      "do_sample": false
    }
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.11.0+cu130"
  },
  "method": "LoRA（量子化なし）",
  "dataset_topic": "講座の用語集",
  "answer_style": "自然文",
  "model_receives_reference_at_inference": false,
  "dataset": {
    "terms": 58,
    "train_examples": 116,
    "evaluation_questions": 15,
    "input_sha256": {
      "glossary.json": "4b614a08a754bf6cf714f905f185ad5ef22e0957c7e64434a4b418edb68c2319",
      "glossary_sft.jsonl": "af6ac820e0331d8d9af376e3b4cccff1b31f2f0f8bb462839752e3282bbd1081",
      "evaluation.json": "2b068446ed4ea335e0e3fa6d98d0eecd23a762b5c0b6e84c6fd2e2cf41298286",
      "glossary